In [1]:
import numpy as np
import pandas as pd

**Goal** Perform and Inner join on the data set with the citations and data set with journal scores. Clean up both data sets as much as possible to join as many rows as possible

In [2]:
df = pd.read_csv('G:\\My Drive\\BrainStation\\BrainStationCapstone\\Scripts and Notebooks\\Cleaned Data\\2023-03-04 Journal Data Frame Cleaning V1.csv')

In [3]:
df.head()

,id,Title,Type,SJR,H index,Country,Region
0,0,nature reviews materials,journal,23.876,131,United Kingdom,Western Europe
1,1,nature energy,journal,16.736,160,United States,Northern America
2,2,nature materials,journal,12.229,507,United Kingdom,Western Europe
3,3,nature nanotechnology,journal,11.698,368,United Kingdom,Western Europe
4,4,nature photonics,journal,10.991,339,United Kingdom,Western Europe


In [4]:
df['id'].tail()

6516    6517
6517    6518
6518    6519
6519    6520
6520    6521
Name: id, dtype: int64

---

Try joining new data frame with the dataset

In [5]:
paper_df = pd.read_csv(r"G:\My Drive\BrainStation\BrainStationCapstone\Scripts and Notebooks\Cleaned Data\2023-03-03 Joining Journal DF out.csv", low_memory = True).drop('Unnamed: 0', axis = 1)

In [6]:
paper_df.head()

,Cites,Title,Year,Source,GSRank,Type,ECC,CitesPerYear,CitesPerAuthor,AuthorCount,query_id,first_auth,second_auth,third_auth
0,4186,low trap-state density and long carrier diffus...,2015.0,science,1,Journal article,4186,523.25,523,8,0.0,d shi,v adinolfi,r comin
1,2319,perovskite light-emitting diodes with external...,2018.0,nature,2,Journal article,2319,463.80,258,9,0.0,k lin,j xing,ln quan
2,2163,solution-processed pbs quantum dot infrared ph...,2005.0,nature materials,3,Journal article,2163,120.17,309,7,0.0,sa mcdonald,g konstantatos,s zhang
3,1951,efficient and stable solution-processed planar...,2017.0,science,4,Journal article,1951,325.17,279,7,0.0,h tan,a jain,o voznyy
4,1897,ultrasensitive solution-cast quantum dot photo...,2006.0,nature,5,Journal article,1897,111.59,271,7,0.0,g konstantatos,i howard,a fischer


In [7]:
join_df1 = pd.merge(left = paper_df, right = df, left_on = 'Source', right_on = 'Title')
join_df1.head()

before = paper_df.shape[0]
after = join_df1.shape[0]

print("{:.0%} of the rows merged!".format(after/before))

85% of the rows merged!


Got 85% of rows merged, but still need about 10% more to reach 95% goal.

In [8]:
jour_df = df

In [9]:
jour_df.head()

,id,Title,Type,SJR,H index,Country,Region
0,0,nature reviews materials,journal,23.876,131,United Kingdom,Western Europe
1,1,nature energy,journal,16.736,160,United States,Northern America
2,2,nature materials,journal,12.229,507,United Kingdom,Western Europe
3,3,nature nanotechnology,journal,11.698,368,United Kingdom,Western Europe
4,4,nature photonics,journal,10.991,339,United Kingdom,Western Europe


In [10]:
def genjoin(replace_dic = None):
    
    global paper_df, jour_df

    if replace_dic != None:
        # replace the words in the original data frame
        paper_df = paper_df.replace(replace_dic)
    
    # Perform Left Join
    left_join = pd.merge(left = paper_df, right = jour_df, left_on = 'Source', right_on = 'Title', how = 'left')
    
    # Find Null Values
    nan_idx = left_join['id'][left_join['id'].isna() == True].index
    nan_df = left_join.loc[nan_idx, :]
    
    print(f"There are {left_join['id'].isna().sum().sum()} Remaining Nulls")
    
    join_df1 = pd.merge(left = paper_df, right = jour_df, left_on = 'Source', right_on = 'Title')
    join_df1.head()

    before = paper_df.shape[0]
    after = join_df1.shape[0]

    print("{:.3%} of the rows merged!".format(after/before))

    return nan_df['Source'].value_counts()

Need to reformat the `paper_df` so that there is a space after `&`

In [11]:
paper_df = paper_df.assign(Source = paper_df.apply(lambda x: x['Source'].replace('&', '& '), axis = 1))
paper_df = paper_df.assign(Source = paper_df.apply(lambda x: x['Source'].replace('-', ''), axis = 1))
paper_df.head()

,Cites,Title,Year,Source,GSRank,Type,ECC,CitesPerYear,CitesPerAuthor,AuthorCount,query_id,first_auth,second_auth,third_auth
0,4186,low trap-state density and long carrier diffus...,2015.0,science,1,Journal article,4186,523.25,523,8,0.0,d shi,v adinolfi,r comin
1,2319,perovskite light-emitting diodes with external...,2018.0,nature,2,Journal article,2319,463.80,258,9,0.0,k lin,j xing,ln quan
2,2163,solution-processed pbs quantum dot infrared ph...,2005.0,nature materials,3,Journal article,2163,120.17,309,7,0.0,sa mcdonald,g konstantatos,s zhang
3,1951,efficient and stable solution-processed planar...,2017.0,science,4,Journal article,1951,325.17,279,7,0.0,h tan,a jain,o voznyy
4,1897,ultrasensitive solution-cast quantum dot photo...,2006.0,nature,5,Journal article,1897,111.59,271,7,0.0,g konstantatos,i howard,a fischer


In [12]:
genjoin()[genjoin() > 10]

There are 2947 Remaining Nulls
87.971% of the rows merged!
There are 2947 Remaining Nulls
87.971% of the rows merged!


journal of materials chemistry                                               102
bulletin of the american physical society                                     97
journal of nanoscience and nanotechnology                                     66
applied physics a                                                             56
chemical physics letters                                                      54
japanese journal of applied physics                                           36
acta crystallographica afoundation and advances                               30
journal of vacuum science & technology b microelectronics and nanometer …     23
laser & photonics reviews                                                     22
inorganic chemistry communications                                            20
materials science and engineering b                                           19
cleo qels_fundamental science                                                 18
journal of the korean magnet

In [13]:
def find_word(row, word):
    result = np.char.find(row['Title'], word)
    if result != -1:
        print(result, row['Title'])
        return True
    elif result > -1:
        return False
    
def find_word2(row, word):
    result = np.char.find(row['Source'], word)
    if result != -1:
        print(result, row['Source'])
        return True
    elif result > -1:
        return False

Remove white space at the end of the `Source` name

In [14]:
paper_df = paper_df.assign(Source = paper_df.apply(lambda x: x['Source'].strip(), axis = 1))

In [15]:
genjoin()

There are 2710 Remaining Nulls
88.939% of the rows merged!


journal of materials chemistry                     102
bulletin of the american physical society           97
journal of nanoscience and nanotechnology           66
applied physics a                                   59
acta crystallographica afoundation and advances     30
                                                  ... 
inżynieria materiałowa                               1
solid state ionics ii volume                         1
évolution                                            1
fuel cells (online)                                  1
journal of the indian botanical society              1
Name: Source, Length: 1062, dtype: int64

In [16]:
genjoin({'applied physics a':'applied physics a materials science and processing'})
#genjoin({'applied physics a ':'applied physics a materials science and processing'})

There are 2651 Remaining Nulls
89.180% of the rows merged!


journal of materials chemistry                                               102
bulletin of the american physical society                                     97
journal of nanoscience and nanotechnology                                     66
acta crystallographica afoundation and advances                               30
journal of vacuum science & technology b microelectronics and nanometer …     23
                                                                            ... 
inżynieria materiałowa                                                         1
solid state ionics ii volume                                                   1
évolution                                                                      1
fuel cells (online)                                                            1
journal of the indian botanical society                                        1
Name: Source, Length: 1061, dtype: int64

In [17]:
genjoin().index

There are 2651 Remaining Nulls
89.180% of the rows merged!


Index(['journal of materials chemistry',
       'bulletin of the american physical society',
       'journal of nanoscience and nanotechnology',
       'acta crystallographica afoundation and advances',
       'journal of vacuum science & technology b microelectronics and nanometer …',
       'laser & photonics reviews', 'inorganic chemistry communications',
       'materials science and engineering b', 'cleo qels_fundamental science',
       'journal of the korean magnetics society',
       ...
       'nano lett', 'metallurgical and materials transactions a',
       'international journal of powder metallurgy',
       'international journal of engineering research & science',
       'tecnología y desarrollo', 'inżynieria materiałowa',
       'solid state ionics ii volume', 'évolution', 'fuel cells (online)',
       'journal of the indian botanical society'],
      dtype='object', length=1061)

In [18]:
genjoin({'acta crystallographica afoundation and advances':'acta crystallographica section a foundations and advances'})

There are 2621 Remaining Nulls
89.302% of the rows merged!


journal of materials chemistry                                               102
bulletin of the american physical society                                     97
journal of nanoscience and nanotechnology                                     66
journal of vacuum science & technology b microelectronics and nanometer …     23
laser & photonics reviews                                                     22
                                                                            ... 
inżynieria materiałowa                                                         1
solid state ionics ii volume                                                   1
évolution                                                                      1
fuel cells (online)                                                            1
journal of the indian botanical society                                        1
Name: Source, Length: 1060, dtype: int64

In [19]:
jour_df.apply(find_word, word = 'photonics reviews', axis = 1);

10 laser and photonics reviews


In [20]:
genjoin({'laser & photonics reviews':'laser and photonics reviews'})

There are 2599 Remaining Nulls
89.392% of the rows merged!


journal of materials chemistry                                               102
bulletin of the american physical society                                     97
journal of nanoscience and nanotechnology                                     66
journal of vacuum science & technology b microelectronics and nanometer …     23
inorganic chemistry communications                                            20
                                                                            ... 
inżynieria materiałowa                                                         1
solid state ionics ii volume                                                   1
évolution                                                                      1
fuel cells (online)                                                            1
journal of the indian botanical society                                        1
Name: Source, Length: 1059, dtype: int64

Let's focus on the journals that have more than 10 citations.

In [21]:
genjoin()[genjoin() > 10]

There are 2599 Remaining Nulls
89.392% of the rows merged!
There are 2599 Remaining Nulls
89.392% of the rows merged!


journal of materials chemistry                                               102
bulletin of the american physical society                                     97
journal of nanoscience and nanotechnology                                     66
journal of vacuum science & technology b microelectronics and nanometer …     23
inorganic chemistry communications                                            20
materials science and engineering b                                           19
journal of the korean magnetics society                                       18
cleo qels_fundamental science                                                 18
journal of applied crystallography                                            17
cleo science and innovations                                                  17
verhandlungen der deutschen physikalischen gesellschaft                       16
infomat                                                                       16
communications materials    

In [22]:
jour_df.apply(find_word, word = 'inorganic chemistry', axis = 1);

0 inorganic chemistry communication
19 russian journal of inorganic chemistry
0 inorganic chemistry frontiers
0 inorganic chemistry
3 bioinorganic chemistry and applications
12 comments on inorganic chemistry
22 journal of biological inorganic chemistry
20 european journal of inorganic chemistry
12 advances in inorganic chemistry
11 reviews in inorganic chemistry
19 chinese journal of inorganic chemistry


In [23]:
genjoin({'inorganic chemistry communications':'inorganic chemistry communication'})

There are 2579 Remaining Nulls
89.474% of the rows merged!


journal of materials chemistry                                               102
bulletin of the american physical society                                     97
journal of nanoscience and nanotechnology                                     66
journal of vacuum science & technology b microelectronics and nanometer …     23
materials science and engineering b                                           19
                                                                            ... 
inżynieria materiałowa                                                         1
solid state ionics ii volume                                                   1
évolution                                                                      1
fuel cells (online)                                                            1
journal of the indian botanical society                                        1
Name: Source, Length: 1058, dtype: int64

In [24]:
jour_df.apply(find_word, word = 'materials science and engineering', axis = 1);

0 materials science and engineering r reports
0 materials science and engineering c
7 acs biomaterials science and engineering
28 modelling and simulation in materials science and engineering
0 materials science and engineering b solidstate materials for advanced technology
12 advances in materials science and engineering
12 archives of materials science and engineering
22 iop conference series materials science and engineering
39 international journal of computational materials science and engineering
19 iranian journal of materials science and engineering
39 fenmo yejin cailiao kexue yu gongcheng/materials science and engineering of powder metallurgy
46 gaofenzi cailiao kexue yu gongcheng/polymeric materials science and engineering


In [25]:
genjoin({'materials science and engineering b':'materials science and engineering b solidstate materials for advanced technology'})

There are 2560 Remaining Nulls
89.552% of the rows merged!


journal of materials chemistry                                               102
bulletin of the american physical society                                     97
journal of nanoscience and nanotechnology                                     66
journal of vacuum science & technology b microelectronics and nanometer …     23
cleo qels_fundamental science                                                 18
                                                                            ... 
inżynieria materiałowa                                                         1
solid state ionics ii volume                                                   1
évolution                                                                      1
fuel cells (online)                                                            1
journal of the indian botanical society                                        1
Name: Source, Length: 1057, dtype: int64

In [26]:
jour_df.apply(find_word, word = 'journal of vacuum science', axis = 1);

0 journal of vacuum science and technology a vacuum, surfaces and films
0 journal of vacuum science and technology bnanotechnology and microelectronics
31 zhenkong kexue yu jishu xuebao/journal of vacuum science and technology


In [27]:
genjoin({'journal of vacuum science &technology b microelectronics and nanometer …':'journal of vacuum science and technology bnanotechnology and microelectronics'})

There are 2560 Remaining Nulls
89.552% of the rows merged!


journal of materials chemistry                                               102
bulletin of the american physical society                                     97
journal of nanoscience and nanotechnology                                     66
journal of vacuum science & technology b microelectronics and nanometer …     23
cleo qels_fundamental science                                                 18
                                                                            ... 
inżynieria materiałowa                                                         1
solid state ionics ii volume                                                   1
évolution                                                                      1
fuel cells (online)                                                            1
journal of the indian botanical society                                        1
Name: Source, Length: 1057, dtype: int64

In [28]:
paper_df.apply(find_word2, word = 'journal of vacuum science', axis = 1)

0 journal of vacuum science & technology b microelectronics and nanometer …
0 journal of vacuum science & technology b microelectronics and nanometer …
0 journal of vacuum science and technology a vacuum, surfaces and films
0 journal of vacuum science & technology b
0 journal of vacuum science and technology a vacuum, surfaces and films
0 journal of vacuum science & technology b microelectronics and nanometer …
0 journal of vacuum science & technology b microelectronics and nanometer …
0 journal of vacuum science & technology b microelectronics and nanometer …
0 journal of vacuum science & technology b microelectronics and nanometer …
0 journal of vacuum science & technology b microelectronics and nanometer …
0 journal of vacuum science & technology b microelectronics and nanometer …
0 journal of vacuum science & technology b microelectronics and nanometer …
0 journal of vacuum science & technology b microelectronics and nanometer …
0 journal of vacuum science & technology b microelect

0        None
1        None
2        None
3        None
4        None
         ... 
24477    None
24478    None
24479    None
24480    None
24481    None
Length: 24482, dtype: object

In [29]:
genjoin({'journal of vacuum science & technology b':'journal of vacuum science and technology bnanotechnology and microelectronics'})

There are 2547 Remaining Nulls
89.605% of the rows merged!


journal of materials chemistry                                               102
bulletin of the american physical society                                     97
journal of nanoscience and nanotechnology                                     66
journal of vacuum science & technology b microelectronics and nanometer …     23
journal of the korean magnetics society                                       18
                                                                            ... 
inżynieria materiałowa                                                         1
solid state ionics ii volume                                                   1
évolution                                                                      1
fuel cells (online)                                                            1
journal of the indian botanical society                                        1
Name: Source, Length: 1056, dtype: int64

In [30]:
genjoin({'journal of vacuum science and technologysection b':'journal of vacuum science and technology bnanotechnology and microelectronics'})

There are 2545 Remaining Nulls
89.613% of the rows merged!


journal of materials chemistry                                               102
bulletin of the american physical society                                     97
journal of nanoscience and nanotechnology                                     66
journal of vacuum science & technology b microelectronics and nanometer …     23
journal of the korean magnetics society                                       18
                                                                            ... 
inżynieria materiałowa                                                         1
solid state ionics ii volume                                                   1
évolution                                                                      1
fuel cells (online)                                                            1
journal of the indian botanical society                                        1
Name: Source, Length: 1055, dtype: int64

In [31]:
genjoin({'journal of vacuum science & technology b microelectronics and nanometer …':'journal of vacuum science and technology bnanotechnology and microelectronics'})

There are 2545 Remaining Nulls
89.613% of the rows merged!


journal of materials chemistry                                               102
bulletin of the american physical society                                     97
journal of nanoscience and nanotechnology                                     66
journal of vacuum science & technology b microelectronics and nanometer …     23
journal of the korean magnetics society                                       18
                                                                            ... 
inżynieria materiałowa                                                         1
solid state ionics ii volume                                                   1
évolution                                                                      1
fuel cells (online)                                                            1
journal of the indian botanical society                                        1
Name: Source, Length: 1055, dtype: int64

Reassigning `journal of vacuum science & technology b microelectronics and nanometer …` is not working. Gonna find the indexes and reassign that way.

In [32]:
paper_df.apply(find_word2, word = '& technology b', axis = 1)

26 journal of vacuum science & technology b microelectronics and nanometer …
26 journal of vacuum science & technology b microelectronics and nanometer …
26 journal of vacuum science & technology b microelectronics and nanometer …
26 journal of vacuum science & technology b microelectronics and nanometer …
26 journal of vacuum science & technology b microelectronics and nanometer …
26 journal of vacuum science & technology b microelectronics and nanometer …
26 journal of vacuum science & technology b microelectronics and nanometer …
26 journal of vacuum science & technology b microelectronics and nanometer …
26 journal of vacuum science & technology b microelectronics and nanometer …
26 journal of vacuum science & technology b microelectronics and nanometer …
26 journal of vacuum science & technology b microelectronics and nanometer …
26 journal of vacuum science & technology b microelectronics and nanometer …
26 journal of vacuum science & technology b microelectronics and nanometer …

0        None
1        None
2        None
3        None
4        None
         ... 
24477    None
24478    None
24479    None
24480    None
24481    None
Length: 24482, dtype: object

The string `'& technology b'` matches the target values. Let's apply this.

In [33]:
# Find where the string '& technology b' is
result = paper_df.apply(lambda row: np.char.find(row['Source'], '& technology b'), axis = 1)

# Return indices where this is true
res_idx = result[result != -1].index

# Reassigning Values
for i in res_idx:
    paper_df.loc[i, 'Source'] = 'journal of vacuum science and technology bnanotechnology and microelectronics'

Check if over writing worked.

In [34]:
genjoin()

There are 2522 Remaining Nulls
89.707% of the rows merged!


journal of materials chemistry               102
bulletin of the american physical society     97
journal of nanoscience and nanotechnology     66
journal of the korean magnetics society       18
cleo qels_fundamental science                 18
                                            ... 
inżynieria materiałowa                         1
solid state ionics ii volume                   1
évolution                                      1
fuel cells (online)                            1
journal of the indian botanical society        1
Name: Source, Length: 1054, dtype: int64

It worked! Let's continue to the other journals.

In [35]:
genjoin()[genjoin() > 10]

There are 2522 Remaining Nulls
89.707% of the rows merged!
There are 2522 Remaining Nulls
89.707% of the rows merged!


journal of materials chemistry                                     102
bulletin of the american physical society                           97
journal of nanoscience and nanotechnology                           66
journal of the korean magnetics society                             18
cleo qels_fundamental science                                       18
cleo science and innovations                                        17
journal of applied crystallography                                  17
verhandlungen der deutschen physikalischen gesellschaft             16
communications materials                                            16
infomat                                                             16
optical nanostructures and advanced materials for photovoltaics     15
journal of materials science & technology                           15
advanced photonics research                                         14
zeitschrift für naturforschung b                                    14
infrar

In [36]:
genjoin({'journal of the korean magnetics society':'journal of magnetics'})

There are 2504 Remaining Nulls
89.780% of the rows merged!


journal of materials chemistry               102
bulletin of the american physical society     97
journal of nanoscience and nanotechnology     66
cleo qels_fundamental science                 18
cleo science and innovations                  17
                                            ... 
tecnología y desarrollo                        1
inżynieria materiałowa                         1
solid state ionics ii volume                   1
évolution                                      1
journal of the indian botanical society        1
Name: Source, Length: 1053, dtype: int64

In [37]:
genjoin()[genjoin() > 10]

There are 2504 Remaining Nulls
89.780% of the rows merged!
There are 2504 Remaining Nulls
89.780% of the rows merged!


journal of materials chemistry                                     102
bulletin of the american physical society                           97
journal of nanoscience and nanotechnology                           66
cleo qels_fundamental science                                       18
cleo science and innovations                                        17
journal of applied crystallography                                  17
communications materials                                            16
infomat                                                             16
verhandlungen der deutschen physikalischen gesellschaft             16
journal of materials science & technology                           15
optical nanostructures and advanced materials for photovoltaics     15
zeitschrift für naturforschung b                                    14
advanced photonics research                                         14
infrared physics & technology                                       13
fronti

In [38]:
jour_df.apply(find_word, word = 'acta crystallographica', axis = 1);

0 acta crystallographica section a foundations and advances
0 acta crystallographica section b structural science, crystal engineering and materials
0 acta crystallographica section e crystallographic communications
0 acta crystallographica. section c, structural chemistry
0 acta crystallographica section fstructural biology communications


In [39]:
genjoin({'acta crystallographica section b structural science':'acta crystallographica section b structural science, crystal engineering and materials'})

There are 2493 Remaining Nulls
89.825% of the rows merged!


journal of materials chemistry               102
bulletin of the american physical society     97
journal of nanoscience and nanotechnology     66
cleo qels_fundamental science                 18
cleo science and innovations                  17
                                            ... 
tecnología y desarrollo                        1
inżynieria materiałowa                         1
solid state ionics ii volume                   1
évolution                                      1
journal of the indian botanical society        1
Name: Source, Length: 1052, dtype: int64

In [40]:
jour_df.apply(find_word, word = 'journal of polymer science', axis = 1);

8 chinese journal of polymer science (english edition)
0 journal of polymer science, part b polymer physics
0 journal of polymer science
14 international journal of polymer science
8 iranian journal of polymer science and technology


In [41]:
genjoin({'journal of polymer science part b polymer physics':'journal of polymer science, part b polymer physics'})

There are 2480 Remaining Nulls
89.878% of the rows merged!


journal of materials chemistry               102
bulletin of the american physical society     97
journal of nanoscience and nanotechnology     66
cleo qels_fundamental science                 18
cleo science and innovations                  17
                                            ... 
tecnología y desarrollo                        1
inżynieria materiałowa                         1
solid state ionics ii volume                   1
évolution                                      1
journal of the indian botanical society        1
Name: Source, Length: 1051, dtype: int64

In [42]:
jour_df.apply(find_word, word = 'journal of materials science', axis = 1);

0 journal of materials science and technology
0 journal of materials science
0 journal of materials science materials in medicine
0 journal of materials science materials in electronics
8 iranian journal of materials science and engineering


In [43]:
genjoin({'journal of materials science & technology':'journal of materials science and technology'})

There are 2465 Remaining Nulls
89.940% of the rows merged!


journal of materials chemistry               102
bulletin of the american physical society     97
journal of nanoscience and nanotechnology     66
cleo qels_fundamental science                 18
cleo science and innovations                  17
                                            ... 
tecnología y desarrollo                        1
inżynieria materiałowa                         1
solid state ionics ii volume                   1
évolution                                      1
journal of the indian botanical society        1
Name: Source, Length: 1050, dtype: int64

In [44]:
jour_df.apply(find_word, word = 'infrared physics', axis = 1);

0 infrared physics and technology


In [45]:
genjoin({'infrared physics & technology':'infrared physics and technology'})

There are 2452 Remaining Nulls
89.993% of the rows merged!


journal of materials chemistry               102
bulletin of the american physical society     97
journal of nanoscience and nanotechnology     66
cleo qels_fundamental science                 18
journal of applied crystallography            17
                                            ... 
tecnología y desarrollo                        1
inżynieria materiałowa                         1
solid state ionics ii volume                   1
évolution                                      1
journal of the indian botanical society        1
Name: Source, Length: 1049, dtype: int64

In [46]:
jour_df.apply(find_word, word = 'advanced materials', axis = 1);

0 advanced materials
0 advanced materials technologies
26 science and technology of advanced materials
19 journal of science advanced materials and devices
13 mechanics of advanced materials and structures
11 reviews on advanced materials science
31 journal of optoelectronics and advanced materials
20 optoelectronics and advanced materials, rapid communications
0 advanced materials and processes
0 advanced materials interfaces


In [47]:
genjoin({'science of advanced materials':'science and technology of advanced materials'})

There are 2440 Remaining Nulls
90.042% of the rows merged!


journal of materials chemistry               102
bulletin of the american physical society     97
journal of nanoscience and nanotechnology     66
cleo qels_fundamental science                 18
journal of applied crystallography            17
                                            ... 
tecnología y desarrollo                        1
inżynieria materiałowa                         1
solid state ionics ii volume                   1
évolution                                      1
journal of the indian botanical society        1
Name: Source, Length: 1048, dtype: int64

In [48]:
jour_df.apply(find_word, word = 'catalysis science', axis = 1);

0 catalysis science and technology


In [49]:
genjoin({'catalysis science & technology':'catalysis science and technology'})

There are 2428 Remaining Nulls
90.091% of the rows merged!


journal of materials chemistry               102
bulletin of the american physical society     97
journal of nanoscience and nanotechnology     66
cleo qels_fundamental science                 18
journal of applied crystallography            17
                                            ... 
tecnología y desarrollo                        1
inżynieria materiałowa                         1
solid state ionics ii volume                   1
évolution                                      1
journal of the indian botanical society        1
Name: Source, Length: 1047, dtype: int64

In [50]:
jour_df.apply(find_word, word = 'c superconductivity', axis = 1);

8 physica c superconductivity and its applications


In [51]:
genjoin({'physica c superconductivity':'physica c superconductivity and its applications'})

There are 2417 Remaining Nulls
90.136% of the rows merged!


journal of materials chemistry               102
bulletin of the american physical society     97
journal of nanoscience and nanotechnology     66
cleo qels_fundamental science                 18
journal of applied crystallography            17
                                            ... 
inżynieria materiałowa                         1
solid state ionics ii volume                   1
évolution                                      1
fuel cells (online)                            1
journal of the indian botanical society        1
Name: Source, Length: 1046, dtype: int64

In [52]:
genjoin()[genjoin() > 10]

There are 2417 Remaining Nulls
90.136% of the rows merged!
There are 2417 Remaining Nulls
90.136% of the rows merged!


journal of materials chemistry                                     102
bulletin of the american physical society                           97
journal of nanoscience and nanotechnology                           66
cleo qels_fundamental science                                       18
journal of applied crystallography                                  17
cleo science and innovations                                        17
communications materials                                            16
infomat                                                             16
verhandlungen der deutschen physikalischen gesellschaft             16
optical nanostructures and advanced materials for photovoltaics     15
advanced photonics research                                         14
zeitschrift für naturforschung b                                    14
frontiers in optics                                                 13
applied catalysis                                                   12
한국자기학회

In [53]:
jour_df.apply(find_word, word = 'applied catalysis', axis = 1);

0 applied catalysis b environmental
0 applied catalysis a general


Need to find which version of `applied catalysis` the journals are

In [54]:
paper_df[paper_df['Source'] == 'applied catalysis']['Title'].values

array(['effect of steam on the coking of platinum catalysts: i. inhibiting effect of steam at low partial pressure for the dehydrogenation of cyclopentane and the coking reaction',
       'characterization of copper-zinc catalysts by hydrogen thermodesorption. correlation with activity in methanol synthesis.',
       'intermetallic compounds as heterogeneous catalysts',
       'effect of partial reduction on the formation of oxygenated compounds in the hydrogenation of carbon monoxide on rhodium/alumina',
       'the role of the metal/support interface in toluene steam reforming over rhodium-alumina catalysts',
       'effect of steam on the coking of platinum catalysts: ii. kinetics',
       'characterization of nickel catalysts by dymanic volumetry and magnetic methods',
       'effect of sulphur on the coking of rhodium in the steam reforming of 1-methylnaphthalene',
       'evaluation of a method for characterizing coked and sulphided rh/al2o3 catalysts: study of the space and time

Found that `applied catalysis` was split into the two versions, the older version is not in the online database, so we'll delete those papers from the data frame.

In [55]:
idx = paper_df[paper_df['Source'] == 'applied catalysis'].index
paper_df = paper_df.drop(idx)

In [56]:
genjoin()[genjoin() > 10]

There are 2405 Remaining Nulls
90.180% of the rows merged!
There are 2405 Remaining Nulls
90.180% of the rows merged!


journal of materials chemistry                                     102
bulletin of the american physical society                           97
journal of nanoscience and nanotechnology                           66
cleo qels_fundamental science                                       18
cleo science and innovations                                        17
journal of applied crystallography                                  17
communications materials                                            16
verhandlungen der deutschen physikalischen gesellschaft             16
infomat                                                             16
optical nanostructures and advanced materials for photovoltaics     15
zeitschrift für naturforschung b                                    14
advanced photonics research                                         14
frontiers in optics                                                 13
한국자기학회 학술연구발표회 논문개요집                                                12
boletí

In [57]:
genjoin()[genjoin() > 10]

There are 2405 Remaining Nulls
90.180% of the rows merged!
There are 2405 Remaining Nulls
90.180% of the rows merged!


journal of materials chemistry                                     102
bulletin of the american physical society                           97
journal of nanoscience and nanotechnology                           66
cleo qels_fundamental science                                       18
cleo science and innovations                                        17
journal of applied crystallography                                  17
communications materials                                            16
verhandlungen der deutschen physikalischen gesellschaft             16
infomat                                                             16
optical nanostructures and advanced materials for photovoltaics     15
zeitschrift für naturforschung b                                    14
advanced photonics research                                         14
frontiers in optics                                                 13
한국자기학회 학술연구발표회 논문개요집                                                12
boletí

In [58]:
jour_df.apply(find_word, word = 'bolet', axis = 1);

0 boletin de la sociedad espanola de ceramica y vidrio
0 boletin de la asociacion de geografos espanoles
0 boletin de la asociacion espanola de entomologia
0 boletin de investigaciones marinas y costeras
0 boletin academico


In [59]:
genjoin({'boletín de la sociedad española de cerámica y vidrio':'boletin de la sociedad espanola de ceramica y vidrio'})

There are 2394 Remaining Nulls
90.225% of the rows merged!


journal of materials chemistry               102
bulletin of the american physical society     97
journal of nanoscience and nanotechnology     66
cleo qels_fundamental science                 18
cleo science and innovations                  17
                                            ... 
inżynieria materiałowa                         1
solid state ionics ii volume                   1
évolution                                      1
fuel cells (online)                            1
journal of the indian botanical society        1
Name: Source, Length: 1044, dtype: int64

In [60]:
genjoin()[genjoin() > 10]

There are 2394 Remaining Nulls
90.225% of the rows merged!
There are 2394 Remaining Nulls
90.225% of the rows merged!


journal of materials chemistry                                     102
bulletin of the american physical society                           97
journal of nanoscience and nanotechnology                           66
cleo qels_fundamental science                                       18
cleo science and innovations                                        17
journal of applied crystallography                                  17
verhandlungen der deutschen physikalischen gesellschaft             16
infomat                                                             16
communications materials                                            16
optical nanostructures and advanced materials for photovoltaics     15
advanced photonics research                                         14
zeitschrift für naturforschung b                                    14
frontiers in optics                                                 13
한국자기학회 학술연구발표회 논문개요집                                                12
journa

In [61]:
jour_df.apply(find_word, word = 'zeitschrift', axis = 1);

0 zeitschrift fur kristallographie crystalline materials
0 zeitschrift fur kristallographie new crystal structures
0 zeitschrift kunststofftechnik/journal of plastics technology
11 keramische zeitschrift
5 zamm zeitschrift fur angewandte mathematik und mechanik
4 zwf zeitschrift fuer wirtschaftlichen fabrikbetrieb
0 zeitschrift fur physikalische chemie
0 zeitschrift fur anorganische und allgemeine chemie
0 zeitschrift fur naturforschung section a journal of physical sciences
0 zeitschrift fur naturforschung section b journal of chemical sciences
0 zeitschrift fur angewandte mathematik und physik


In [62]:
genjoin({'zeitschrift für naturforschung b':'zeitschrift fur naturforschung section b journal of chemical sciences'})

There are 2380 Remaining Nulls
90.282% of the rows merged!


journal of materials chemistry               102
bulletin of the american physical society     97
journal of nanoscience and nanotechnology     66
cleo qels_fundamental science                 18
journal of applied crystallography            17
                                            ... 
inżynieria materiałowa                         1
solid state ionics ii volume                   1
évolution                                      1
fuel cells (online)                            1
journal of the indian botanical society        1
Name: Source, Length: 1043, dtype: int64

In [63]:
genjoin()[genjoin() > 10]

There are 2380 Remaining Nulls
90.282% of the rows merged!
There are 2380 Remaining Nulls
90.282% of the rows merged!


journal of materials chemistry                                     102
bulletin of the american physical society                           97
journal of nanoscience and nanotechnology                           66
cleo qels_fundamental science                                       18
journal of applied crystallography                                  17
cleo science and innovations                                        17
communications materials                                            16
verhandlungen der deutschen physikalischen gesellschaft             16
infomat                                                             16
optical nanostructures and advanced materials for photovoltaics     15
advanced photonics research                                         14
frontiers in optics                                                 13
한국자기학회 학술연구발표회 논문개요집                                                12
journal of physics materials                                        11
Name: 

Need to investigate journal names that can't easily be found in `jour_df`, may need to manually add the entry if it got missed.

In [64]:
paper_df[paper_df['Source'] == 'journal of physics materials']['Title'].values[:2]

array(['observation of long spin lifetime in mapbbr3 single crystals at room temperature',
       'role of defects and phonons in bandgap dynamics of monolayer ws2 at high carrier densities'],
      dtype=object)

In [65]:
jour_df.apply(find_word, word = 'jphys', axis = 1);

0 jphys energy
0 jphys materials
0 jphys photonics


In [66]:
genjoin({'journal of physics materials':'jphys materials'})

There are 2369 Remaining Nulls
90.327% of the rows merged!


journal of materials chemistry               102
bulletin of the american physical society     97
journal of nanoscience and nanotechnology     66
cleo qels_fundamental science                 18
journal of applied crystallography            17
                                            ... 
inżynieria materiałowa                         1
solid state ionics ii volume                   1
évolution                                      1
fuel cells (online)                            1
journal of the indian botanical society        1
Name: Source, Length: 1042, dtype: int64

In [67]:
paper_df[paper_df['Source'] == 'bulletin of the american physical society']['Title'].values[:2]

array(['thermoelectric transport signatures of carrier interactions in polymer electrochemical transistors',
       'new mechanism for work-function tuning: zno surfaces modified by a strong organic electron acceptor'],
      dtype=object)

`bulletin of the american physical society` is not an actual journal. Let's delete these from our database.

In [68]:
idx = paper_df[paper_df['Source'] == 'bulletin of the american physical society'].index
paper_df = paper_df.drop(idx)

In [69]:
genjoin()

There are 2272 Remaining Nulls
90.686% of the rows merged!


journal of materials chemistry               102
journal of nanoscience and nanotechnology     66
cleo qels_fundamental science                 18
cleo science and innovations                  17
journal of applied crystallography            17
                                            ... 
inżynieria materiałowa                         1
solid state ionics ii volume                   1
évolution                                      1
fuel cells (online)                            1
journal of the indian botanical society        1
Name: Source, Length: 1041, dtype: int64

In [70]:
paper_df[paper_df['Source'] == 'jphys']['Title'].values

array([], dtype=object)

Need to manually add `journal of materials chemistry` to `jour_df`

In [71]:
jour_df.tail()

,id,Title,Type,SJR,H index,Country,Region
6516,6517,bulletin de la societe vaudoise des sciences n...,journal,0.101,11,Switzerland,Western Europe
6517,6518,radovi zavoda za znanstvenoistrazivacki i umje...,journal,0.101,1,Croatia,Eastern Europe
6518,6519,taiwan review,journal,0.100,3,Taiwan,Asiatic Region
6519,6520,international journal of advanced and applied ...,journal,NaN,2,Taiwan,Asiatic Region
6520,6521,journal of scientific exploration,journal,NaN,18,United States,Northern America


In [72]:
id_ = jour_df['id'].values[-1] + 1
title = 'journal of materials chemistry'
type_ = 'journal'
SJR = 2.41
H_index = int(314)
country = 'United Kingdom'
region = 'Western Europe'

jour_df.loc[jour_df.index.max() + 1, :] = [id_, title, type_, SJR, H_index, country, region]

In [73]:
jour_df.tail()

,id,Title,Type,SJR,H index,Country,Region
6517,6518.0,radovi zavoda za znanstvenoistrazivacki i umje...,journal,0.101,1.0,Croatia,Eastern Europe
6518,6519.0,taiwan review,journal,0.100,3.0,Taiwan,Asiatic Region
6519,6520.0,international journal of advanced and applied ...,journal,NaN,2.0,Taiwan,Asiatic Region
6520,6521.0,journal of scientific exploration,journal,NaN,18.0,United States,Northern America
6521,6522.0,journal of materials chemistry,journal,2.410,314.0,United Kingdom,Western Europe


In [74]:
genjoin()

There are 2170 Remaining Nulls
91.105% of the rows merged!


journal of nanoscience and nanotechnology    66
cleo qels_fundamental science                18
cleo science and innovations                 17
journal of applied crystallography           17
infomat                                      16
                                             ..
inżynieria materiałowa                        1
solid state ionics ii volume                  1
évolution                                     1
fuel cells (online)                           1
journal of the indian botanical society       1
Name: Source, Length: 1040, dtype: int64

Need to manually add `journal of nanoscience and nanotechnology` to `jour_df`

In [75]:
id_ = jour_df['id'].values[-1] + 1
title = 'journal of nanoscience and nanotechnology'
type_ = 'journal'
SJR = 0.235
H_index = int(108)
country = 'United States'
region = 'Northern America'

jour_df.loc[jour_df.index.max() + 1, :] = [id_, title, type_, SJR, H_index, country, region]

In [76]:
jour_df.tail()

,id,Title,Type,SJR,H index,Country,Region
6518,6519.0,taiwan review,journal,0.100,3.0,Taiwan,Asiatic Region
6519,6520.0,international journal of advanced and applied ...,journal,NaN,2.0,Taiwan,Asiatic Region
6520,6521.0,journal of scientific exploration,journal,NaN,18.0,United States,Northern America
6521,6522.0,journal of materials chemistry,journal,2.410,314.0,United Kingdom,Western Europe
6522,6523.0,journal of nanoscience and nanotechnology,journal,0.235,108.0,United States,Northern America


In [77]:
genjoin()[genjoin() > 10]

There are 2104 Remaining Nulls
91.376% of the rows merged!
There are 2104 Remaining Nulls
91.376% of the rows merged!


cleo qels_fundamental science                                      18
cleo science and innovations                                       17
journal of applied crystallography                                 17
infomat                                                            16
communications materials                                           16
verhandlungen der deutschen physikalischen gesellschaft            16
optical nanostructures and advanced materials for photovoltaics    15
advanced photonics research                                        14
frontiers in optics                                                13
한국자기학회 학술연구발표회 논문개요집                                               12
Name: Source, dtype: int64

In [78]:
paper_df[paper_df['Source'] == 'cleo qels_fundamental science']['Title'].values[:3]

array(['enhanced multiphoton photoluminescence in metaphotonics',
       'strongly coupled hybrid frenkel/wannier-mott exciton polaritons in a high q microcavity',
       'auger-type hole trapping process at green emission centers of zno nanowires'],
      dtype=object)

In [79]:
jour_df.apply(find_word, word = 'cleo', axis = 1);

Need to Manually add `cleo qels_fundamental science`

In [80]:
id_ = jour_df['id'].values[-1] + 1
title = 'cleo qels_fundamental science'
type_ = 'conferences and proceedings'
SJR = 0.112
H_index = int(5)
country = 'United States'
region = 'Northern America'

jour_df.loc[jour_df.index.max() + 1, :] = [id_, title, type_, SJR, H_index, country, region]

In [81]:
genjoin()[genjoin() > 10]

There are 2086 Remaining Nulls
91.450% of the rows merged!
There are 2086 Remaining Nulls
91.450% of the rows merged!


cleo science and innovations                                       17
journal of applied crystallography                                 17
verhandlungen der deutschen physikalischen gesellschaft            16
communications materials                                           16
infomat                                                            16
optical nanostructures and advanced materials for photovoltaics    15
advanced photonics research                                        14
frontiers in optics                                                13
한국자기학회 학술연구발표회 논문개요집                                               12
Name: Source, dtype: int64

In [82]:
paper_df[paper_df['Source'] == 'cleo science and innovations']['Title'].values[:3]

array(['halide-perovskite metasurfaces governed by the kerker effect',
       'blue superluminescent diodes with ghz bandwidth exciting perovskite nanocrystals for high cri white lighting and high-speed vlc',
       'continuous-wave optically pumped lasing of hybrid perovskite vcsel at green wavelength'],
      dtype=object)

`cleo science and innovations` is a conference processing and has different values for each year. Each one of the papers with a `cleo science and innovations` might be from a different year, need to check.

In [83]:
paper_df[paper_df['Source'] == 'cleo science and innovations']['Year'].value_counts()

2015.0    5
2013.0    4
2021.0    3
2019.0    3
2017.0    1
2022.0    1
Name: Year, dtype: int64

In [85]:
paper_df[(paper_df['Source'] == 'cleo science and innovations') & (paper_df['Year'] == 2022)]['Title'].values

array(['low-concentration detection of co2 using suspended silicon waveguides in the mid-ir'],
      dtype=object)

Only can find data base values on www.scimagojr.com from 2012, 2013 and 2015, but its unclear if the papers cited actually are from those journals. Will just delete these papers from `paper_df`

In [86]:
idx = paper_df[paper_df['Source'] == 'cleo science and innovations'].index
paper_df = paper_df.drop(idx)

In [87]:
genjoin()

There are 2069 Remaining Nulls
91.513% of the rows merged!


journal of applied crystallography                                 17
infomat                                                            16
verhandlungen der deutschen physikalischen gesellschaft            16
communications materials                                           16
optical nanostructures and advanced materials for photovoltaics    15
                                                                   ..
inżynieria materiałowa                                              1
solid state ionics ii volume                                        1
évolution                                                           1
fuel cells (online)                                                 1
journal of the indian botanical society                             1
Name: Source, Length: 1037, dtype: int64

In [88]:
paper_df[paper_df['Source'] == 'journal of applied crystallography']['Title'].values[:3]

array(['vapour growth, morphology, absolute structure and pyroelectric coefficient of meta-nitroaniline single crystals',
       'pseudo: a program for a pseudosymmetry search',
       'mode-crystallography analysis of the crystal structures and the low-and high-temperature phase transitions in na0. 5k0. 5nbo3'],
      dtype=object)

Need to manually add `journal of applied crystallography` to `jour_df`

In [89]:
id_ = jour_df['id'].values[-1] + 1
title = 'journal of applied crystallography'
type_ = 'journal'
SJR = 1.385
H_index = int(5)
country = 'United Kingdom'
region = 'Western Europe'

jour_df.loc[jour_df.index.max() + 1, :] = [id_, title, type_, SJR, H_index, country, region]

In [90]:
genjoin()[genjoin() > 10]

There are 2052 Remaining Nulls
91.583% of the rows merged!
There are 2052 Remaining Nulls
91.583% of the rows merged!


verhandlungen der deutschen physikalischen gesellschaft            16
communications materials                                           16
infomat                                                            16
optical nanostructures and advanced materials for photovoltaics    15
advanced photonics research                                        14
frontiers in optics                                                13
한국자기학회 학술연구발표회 논문개요집                                               12
Name: Source, dtype: int64

In [91]:
paper_df[paper_df['Source'] == 'verhandlungen der deutschen physikalischen gesellschaft']['Title'].values[:3]

array(['hole injection barrier optimization at ito/organic interfaces modified with strong molecular acceptors',
       'absorption of {pi}-conjugated molecules on cu (111) studied by x-ray standing waves',
       'polymer-based optoelectronic devices with interlayers'],
      dtype=object)

Can't find `verhandlungen der deutschen physikalischen gesellschaft` in online data base despite the title being correct, will delete the papers from `papers_df`

In [92]:
idx = paper_df[paper_df['Source'] == 'verhandlungen der deutschen physikalischen gesellschaft'].index
paper_df = paper_df.drop(idx)

In [93]:
genjoin()[genjoin() > 10]

There are 2036 Remaining Nulls
91.643% of the rows merged!
There are 2036 Remaining Nulls
91.643% of the rows merged!


communications materials                                           16
infomat                                                            16
optical nanostructures and advanced materials for photovoltaics    15
advanced photonics research                                        14
frontiers in optics                                                13
한국자기학회 학술연구발표회 논문개요집                                               12
Name: Source, dtype: int64

In [94]:
paper_df[paper_df['Source'] == 'communications materials']['Title'].values[:3]

array(['out-of-plane trion emission in monolayer wse2 revealed by whispering gallery modes of dielectric microresonators',
       'multi-functional multi-gate one-transistor process-in-memory electronics with foundry processing and footprint reduction',
       'challenges and strategies toward long-term stability of lead-free tin-based perovskite solar cells'],
      dtype=object)

Couldn't find `communications materials` in the online database, will delete values from `paper_df`

In [95]:
idx = paper_df[paper_df['Source'] == 'communications materials'].index
paper_df = paper_df.drop(idx)

In [96]:
genjoin()[genjoin() > 10]

There are 2020 Remaining Nulls
91.704% of the rows merged!
There are 2020 Remaining Nulls
91.704% of the rows merged!


infomat                                                            16
optical nanostructures and advanced materials for photovoltaics    15
advanced photonics research                                        14
frontiers in optics                                                13
한국자기학회 학술연구발표회 논문개요집                                               12
Name: Source, dtype: int64

In [97]:
paper_df[paper_df['Source'] == 'infomat']['Title'].values[:3]

array(['organicinorganic hybrid perovskite scintillators for mixed field radiation detection',
       'emerging reconfigurable electronic devices based on twodimensional materials: a review',
       'probing the dynamic structural changes of dna using ultrafast laser pulse in graphenebased optofluidic device'],
      dtype=object)

Couldn't find `infomat` in the online database, will delete values from `paper_df`

In [98]:
idx = paper_df[paper_df['Source'] == 'infomat'].index
paper_df = paper_df.drop(idx)

In [99]:
genjoin()[genjoin() > 10]

There are 2004 Remaining Nulls
91.764% of the rows merged!
There are 2004 Remaining Nulls
91.764% of the rows merged!


optical nanostructures and advanced materials for photovoltaics    15
advanced photonics research                                        14
frontiers in optics                                                13
한국자기학회 학술연구발표회 논문개요집                                               12
Name: Source, dtype: int64

In [100]:
paper_df[paper_df['Source'] == 'optical nanostructures and advanced materials for photovoltaics']['Title'].values[:3]

array(['inorganic passivation and doping control in colloidal quantum dot photovoltaics',
       'optical optimization of perovskite-silicon reflective tandem solar cells',
       'optical modelling of mae textured nanoporous silicon'],
      dtype=object)

Couldn't find `optical nanostructures and advanced materials for photovoltaics` in the online database, will delete values from `paper_df`

In [101]:
idx = paper_df[paper_df['Source'] == 'optical nanostructures and advanced materials for photovoltaics'].index
paper_df = paper_df.drop(idx)

In [102]:
genjoin()[genjoin() > 9]

There are 1989 Remaining Nulls
91.821% of the rows merged!
There are 1989 Remaining Nulls
91.821% of the rows merged!


advanced photonics research                            14
frontiers in optics                                    13
한국자기학회 학술연구발표회 논문개요집                                   12
chemistry–an asian journal                             10
proceedings of the national academy of sciences        10
journal of polymer science part a polymer chemistry    10
applied clay science                                   10
acta cryst                                             10
oriental journal of chemistry                          10
organic                                                10
journal of physical chemistry                          10
particle & particle systems characterization           10
microchimica acta                                      10
biophysical journal                                    10
phys. rev. b                                           10
Name: Source, dtype: int64

In [103]:
paper_df[paper_df['Source'] == 'advanced photonics research']['Title'].values[:3]

array(['ultrafast pumpprobe microscopy on 2d transition metal dichalcogenides',
       'enhanced raman scattering for probing nearfield distribution in alldielectric nanostructures',
       'nanophotonics for perovskite solar cells'], dtype=object)

Couldn't find `advanced photonics research` in the online database, will delete values from `paper_df`

In [104]:
idx = paper_df[paper_df['Source'] == 'advanced photonics research'].index
paper_df = paper_df.drop(idx)

In [105]:
genjoin()[genjoin() > 9]

There are 1975 Remaining Nulls
91.874% of the rows merged!
There are 1975 Remaining Nulls
91.874% of the rows merged!


frontiers in optics                                    13
한국자기학회 학술연구발표회 논문개요집                                   12
chemistry–an asian journal                             10
organic                                                10
oriental journal of chemistry                          10
acta cryst                                             10
particle & particle systems characterization           10
microchimica acta                                      10
applied clay science                                   10
phys. rev. b                                           10
journal of polymer science part a polymer chemistry    10
journal of physical chemistry                          10
biophysical journal                                    10
proceedings of the national academy of sciences        10
Name: Source, dtype: int64

In [106]:
paper_df[paper_df['Source'] == 'frontiers in optics']['Title'].values[:3]

array(['depleted-heterojunction colloidal quantum dot solar cells employing low-cost metal contacts',
       'efficient high-order optical harmonics generation from resonant semiconductor metasurfaces supporting bound states in the continuum',
       'multiple exciton generation in silicon nanocrystals'],
      dtype=object)

`frontiers in optics` is a conference processing and has different values for each year. Will delete from data base since like `cleo science and innovations`

In [107]:
idx = paper_df[paper_df['Source'] == 'frontiers in optics'].index
paper_df = paper_df.drop(idx)

In [108]:
genjoin()[genjoin() > 9]

There are 1962 Remaining Nulls
91.923% of the rows merged!
There are 1962 Remaining Nulls
91.923% of the rows merged!


한국자기학회 학술연구발표회 논문개요집                                   12
chemistry–an asian journal                             10
biophysical journal                                    10
organic                                                10
oriental journal of chemistry                          10
acta cryst                                             10
particle & particle systems characterization           10
microchimica acta                                      10
applied clay science                                   10
phys. rev. b                                           10
journal of polymer science part a polymer chemistry    10
journal of physical chemistry                          10
proceedings of the national academy of sciences        10
Name: Source, dtype: int64

In [109]:
paper_df[paper_df['Source'] == '한국자기학회 학술연구발표회 논문개요집']['Title'].values[:3]

array(['room-temperature ferromagnetism and the related electronic properties of sb-doped tio2-',
       'magnetic and electronic properties of reduced rutile ti1-xmnxo2- thin films',
       'magnetic properties of fe doped cupric oxide'], dtype=object)

`한국자기학회 학술연구발표회 논문개요집` is a conference proceeding and has different values for each year. Will delete from data base since like `cleo science and innovations`

In [110]:
idx = paper_df[paper_df['Source'] == '한국자기학회 학술연구발표회 논문개요집'].index
paper_df = paper_df.drop(idx)

In [111]:
genjoin()[genjoin() > 9]

There are 1950 Remaining Nulls
91.968% of the rows merged!
There are 1950 Remaining Nulls
91.968% of the rows merged!


proceedings of the national academy of sciences        10
organic                                                10
phys. rev. b                                           10
oriental journal of chemistry                          10
microchimica acta                                      10
particle & particle systems characterization           10
journal of polymer science part a polymer chemistry    10
journal of physical chemistry                          10
acta cryst                                             10
chemistry–an asian journal                             10
biophysical journal                                    10
applied clay science                                   10
Name: Source, dtype: int64

Will also delete `proceedings of the national academy of sciences` since its a conference proceeding.

In [112]:
idx = paper_df[paper_df['Source'] == 'proceedings of the national academy of sciences'].index
paper_df = paper_df.drop(idx)

In [113]:
genjoin()[genjoin() > 9]

There are 1940 Remaining Nulls
92.006% of the rows merged!
There are 1940 Remaining Nulls
92.006% of the rows merged!


microchimica acta                                      10
chemistry–an asian journal                             10
acta cryst                                             10
applied clay science                                   10
phys. rev. b                                           10
biophysical journal                                    10
oriental journal of chemistry                          10
organic                                                10
particle & particle systems characterization           10
journal of physical chemistry                          10
journal of polymer science part a polymer chemistry    10
Name: Source, dtype: int64

Need to manually add `journal of physical chemistry` to `jour_df`

In [114]:
id_ = jour_df['id'].values[-1] + 1
title = 'journal of physical chemistry'
type_ = 'journal'
SJR = 1.385
H_index = int(170)
country = 'United States'
region = 'Northern America'

jour_df.loc[jour_df.index.max() + 1, :] = [id_, title, type_, SJR, H_index, country, region]

In [115]:
genjoin()[genjoin() > 9]

There are 1930 Remaining Nulls
92.048% of the rows merged!
There are 1930 Remaining Nulls
92.048% of the rows merged!


applied clay science                                   10
organic                                                10
journal of polymer science part a polymer chemistry    10
particle & particle systems characterization           10
chemistry–an asian journal                             10
microchimica acta                                      10
acta cryst                                             10
biophysical journal                                    10
phys. rev. b                                           10
oriental journal of chemistry                          10
Name: Source, dtype: int64

The remaining are likely journal mis-formatting. Will continue as such.

In [116]:
jour_df.apply(find_word, word = 'an asian journal', axis = 1);

10 chemistry an asian journal


In [117]:
genjoin({'chemistry–an asian journal':'chemistry an asian journal'})

There are 1920 Remaining Nulls
92.089% of the rows merged!


microchimica acta                          10
organic                                    10
oriental journal of chemistry              10
acta cryst                                 10
biophysical journal                        10
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 1026, dtype: int64

In [118]:
genjoin()[genjoin() > 9]

There are 1920 Remaining Nulls
92.089% of the rows merged!
There are 1920 Remaining Nulls
92.089% of the rows merged!


microchimica acta                                      10
organic                                                10
oriental journal of chemistry                          10
acta cryst                                             10
biophysical journal                                    10
phys. rev. b                                           10
particle & particle systems characterization           10
journal of polymer science part a polymer chemistry    10
applied clay science                                   10
Name: Source, dtype: int64

In [119]:
jour_df.apply(find_word, word = 'physical rev', axis = 1);

0 physical review b
0 physical review materials
0 physical review fluids
0 physical review x
0 physical review letters
0 physical review d
0 physical review applied
0 physical review c
0 physical review a
0 physical review physics education research
0 physical review accelerators and beams
0 physical review e


In [120]:
genjoin({'phys. rev. b':'physical review b'})

There are 1910 Remaining Nulls
92.130% of the rows merged!


journal of polymer science part a polymer chemistry    10
applied clay science                                   10
particle & particle systems characterization           10
microchimica acta                                      10
oriental journal of chemistry                          10
                                                       ..
inżynieria materiałowa                                  1
solid state ionics ii volume                            1
évolution                                               1
fuel cells (online)                                     1
journal of the indian botanical society                 1
Name: Source, Length: 1025, dtype: int64

In [121]:
genjoin()[genjoin() > 9]

There are 1910 Remaining Nulls
92.130% of the rows merged!
There are 1910 Remaining Nulls
92.130% of the rows merged!


journal of polymer science part a polymer chemistry    10
applied clay science                                   10
particle & particle systems characterization           10
microchimica acta                                      10
oriental journal of chemistry                          10
acta cryst                                             10
biophysical journal                                    10
organic                                                10
Name: Source, dtype: int64

Need to access titles of remaining journals to see if the journal names are in the data base.

In [122]:
paper_df[paper_df['Source'] == 'acta cryst']['Title'].values

array(['photoluminescence in lead halide perovskites and the role of defects',
       'ms74. p28',
       'ms16-05| mead, salt, and sunshine: cation distribution in cztse, cfts, and czsise',
       'ms19-p15 diffraction and xafs-the multi-purpose beamline kmc-2',
       'p. ms. 12p. ms. 95', 'ms74. p28',
       'ms16-05| mead, salt, and sunshine: cation distribution in cztse, cfts, and czsise',
       'ms19-p15 diffraction and xafs-the multi-purpose beamline kmc-2',
       'p. ms. 12p. ms. 95',
       'ms16-p03| structure design of novel ba3-xsrxteo6 double perovskites and the effect of temperature and composition on structure stability'],
      dtype=object)

Besides `photoluminescence in lead halide perovskites and the role of defects` the citations here are from posters. Assigning `photoluminescence in lead halide perovskites and the role of defects` manually and will delete the rest of the citations.

In [123]:
idx = paper_df[paper_df['Title'] == 'photoluminescence in lead halide perovskites and the role of defects'].index
paper_df.loc[idx, 'Source'] = 'acta crystallographica section a foundations and advances'

In [124]:
idx = paper_df[paper_df['Source'] == 'acta cryst'].index
paper_df = paper_df.drop(idx)

In [125]:
genjoin()[genjoin() > 9]

There are 1900 Remaining Nulls
92.168% of the rows merged!
There are 1900 Remaining Nulls
92.168% of the rows merged!


journal of polymer science part a polymer chemistry    10
organic                                                10
particle & particle systems characterization           10
oriental journal of chemistry                          10
applied clay science                                   10
biophysical journal                                    10
microchimica acta                                      10
Name: Source, dtype: int64

In [126]:
jour_df.apply(find_word, word = 'journal of polymer', axis = 1);

8 chinese journal of polymer science (english edition)
0 journal of polymers and the environment
0 journal of polymer science, part b polymer physics
0 journal of polymer science
14 international journal of polymer science
14 international journal of polymeric materials and polymeric biomaterials
0 journal of polymer research
14 international journal of polymer analysis and characterization
0 journal of polymer engineering
8 iranian journal of polymer science and technology


Need to manually add `journal of polymer science part a polymer chemistry` to `jour_df`

In [127]:
id_ = jour_df['id'].values[-1] + 1
title = 'journal of polymer science part a polymer chemistry'
type_ = 'journal'
SJR = 0.768
H_index = int(0)
country = 'United States'
region = 'Northern America'

jour_df.loc[jour_df.index.max() + 1, :] = [id_, title, type_, SJR, H_index, country, region]

In [128]:
genjoin()[genjoin() > 9]

There are 1890 Remaining Nulls
92.210% of the rows merged!
There are 1890 Remaining Nulls
92.210% of the rows merged!


microchimica acta                               10
organic                                         10
particle & particle systems characterization    10
oriental journal of chemistry                   10
biophysical journal                             10
applied clay science                            10
Name: Source, dtype: int64

In [129]:
jour_df.apply(find_word, word = 'particle systems characterization', axis = 1);

13 particle and particle systems characterization


In [130]:
genjoin({'particle & particle systems characterization':'particle and particle systems characterization'})

There are 1880 Remaining Nulls
92.251% of the rows merged!


biophysical journal                        10
applied clay science                       10
microchimica acta                          10
organic                                    10
oriental journal of chemistry              10
                                           ..
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
inzynieria materialowa                      1
journal of the indian botanical society     1
Name: Source, Length: 1022, dtype: int64

In [131]:
jour_df.apply(find_word, word = 'biophys', axis = 1);

28 radiation and environmental biophysics
11 journal of biophysics
57 nato science for peace and security series b physics and biophysics
14 biochimica et biophysica acta proteins and proteomics
5 food biophysics
0 biophysical chemistry
25 doklady biochemistry and biophysics
25 journal of computational biophysics and chemistry
0 biophysical bulletin
17 annual review of biophysics


Need to manually add `biophysical journal`

In [132]:
id_ = jour_df['id'].values[-1] + 1
title = 'biophysical journal'
type_ = 'journal'
SJR = 1.243
H_index = int(278)
country = 'United States'
region = 'Northern America'

jour_df.loc[jour_df.index.max() + 1, :] = [id_, title, type_, SJR, H_index, country, region]

In [133]:
genjoin()[genjoin() > 9]

There are 1870 Remaining Nulls
92.292% of the rows merged!
There are 1870 Remaining Nulls
92.292% of the rows merged!


applied clay science             10
microchimica acta                10
organic                          10
oriental journal of chemistry    10
Name: Source, dtype: int64

In [134]:
paper_df[paper_df['Source'] == 'applied clay science']['Title'].values[:3]

array(['effect of layer charge and charge distribution on the formation of chitosan-smectite bionanocomposites',
       'effect of layer charge and charge distribution on the formation of chitosan-smectite bionanocomposites',
       'physico-mechanical and gas permeability characteristics of kaolin based ceramic membranes prepared with a new pore-forming agent'],
      dtype=object)

In [135]:
jour_df.apply(find_word, word = 'clay', axis = 1);

0 clays and clay minerals


Need to manually add `applied clay science`

In [136]:
id_ = jour_df['id'].values[-1] + 1
title = 'applied clay science'
type_ = 'journal'
SJR = 0.958
H_index = int(142)
country = 'Netherlands'
region = 'Western Europe'

jour_df.loc[jour_df.index.max() + 1, :] = [id_, title, type_, SJR, H_index, country, region]

In [137]:
genjoin()[genjoin() > 9]

There are 1860 Remaining Nulls
92.333% of the rows merged!
There are 1860 Remaining Nulls
92.333% of the rows merged!


oriental journal of chemistry    10
microchimica acta                10
organic                          10
Name: Source, dtype: int64

In [138]:
paper_df[paper_df['Source'] == 'oriental journal of chemistry']['Title'].values[:3]

array(['chromium (iii) and iron (iii) complexes of 3-hydroxy imino-1-n-phenyl amino butane-1-one',
       'iron (11) complexes of triazene-1-oxides',
       'tin (ii) compelxes of triazene-1-oxides'], dtype=object)

Need to manually add `oriental journal of chemistry`

In [139]:
id_ = jour_df['id'].values[-1] + 1
title = 'oriental journal of chemistry'
type_ = 'journal'
SJR = 0.164
H_index = int(27)
country = 'India'
region = 'Asiatic Region'

jour_df.loc[jour_df.index.max() + 1, :] = [id_, title, type_, SJR, H_index, country, region]

In [140]:
genjoin()[genjoin() > 9]

There are 1850 Remaining Nulls
92.375% of the rows merged!
There are 1850 Remaining Nulls
92.375% of the rows merged!


organic              10
microchimica acta    10
Name: Source, dtype: int64

In [141]:
paper_df[paper_df['Source'] == 'organic']['Title'].values[:3]

array(['investigating the effect of lamination on famacs: toward a new phase space of perovskite solar cell fabrication',
       'phase segregation control in mixed halide and mixed cation perovskite films: synergistic effects of cs and rb (conference presentation)',
       'understanding the impact of carrier mobility and mobile ions on perovskite cell performance'],
      dtype=object)

Can't find `Organic, Hybrid, and Perovskite Photovoltaics` in the online data base. Deleting citations from `paper_df`

In [142]:
idx = paper_df[paper_df['Source'] == 'organic'].index
paper_df = paper_df.drop(idx)

In [143]:
genjoin()[genjoin() > 9]

There are 1840 Remaining Nulls
92.413% of the rows merged!
There are 1840 Remaining Nulls
92.413% of the rows merged!


microchimica acta    10
Name: Source, dtype: int64

In [144]:
paper_df[paper_df['Source'] == 'microchimica acta']['Title'].values[:3]

array(['electrodeposition and electrocatalytic properties of platinum nanoparticles on multi-walled carbon nanotubes: effect of the deposition conditions',
       'a carbon paste electrode modified with a nickel titanate nanoceramic for simultaneous voltammetric determination of ortho- and para-hydroxybenzoic acids',
       'a carbon paste electrode modified with a nickel titanate nanoceramic for simultaneous voltammetric determination of ortho- and para-hydroxybenzoic acids'],
      dtype=object)

Can't find `microchimica acta` in the online data base. Deleting citations from `paper_df`

In [145]:
idx = paper_df[paper_df['Source'] == 'microchimica acta'].index
paper_df = paper_df.drop(idx)

Now will move to journals that have more than 5 citations

In [146]:
genjoin()[genjoin() > 5]

There are 1830 Remaining Nulls
92.451% of the rows merged!
There are 1830 Remaining Nulls
92.451% of the rows merged!


journal of molecular catalysis a chemical                                9
journal of physics energy                                                9
optics for solar energy                                                  9
npj                                                                      9
photochemical & photobiological sciences                                 8
physica status solidiaapplied research                                   8
physics procedia                                                         8
physica status solidi (c)                                                8
ecomat                                                                   7
physics letters a                                                        7
ieee transactions on ultrasonics                                         7
acta crystallographica section e structure reports online                7
european physical journalapplied physics                                 7
physica status solidi c  

In [147]:
jour_df.apply(find_word, word = 'physics letters', axis = 1);

9 chemical physics letters
9 chemical physics letters x
0 physics letters, section b nuclear, elementary particle and highenergy physics
8 applied physics letters
0 physics letters, section a general, atomic and solid state physics
4 europhysics letters
6 laser physics letters
8 chinese physics letters
7 modern physics letters a
7 modern physics letters b
10 technical physics letters


In [148]:
genjoin({'physics letters a': 'physics letters, section a general, atomic and solid state physics'})

There are 1823 Remaining Nulls
92.480% of the rows merged!


optics for solar energy                      9
journal of physics energy                    9
npj                                          9
journal of molecular catalysis a chemical    9
physica status solidiaapplied research       8
                                            ..
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
inzynieria materialowa                       1
journal of the indian botanical society      1
Name: Source, Length: 1016, dtype: int64

In [149]:
jour_df.apply(find_word, word = 'ieee transactions on ultrasonics', axis = 1);

0 ieee transactions on ultrasonics, ferroelectrics, and frequency control


In [150]:
genjoin({'ieee transactions on ultrasonics': 'ieee transactions on ultrasonics, ferroelectrics, and frequency control'})

There are 1816 Remaining Nulls
92.509% of the rows merged!


journal of molecular catalysis a chemical    9
optics for solar energy                      9
npj                                          9
journal of physics energy                    9
physics procedia                             8
                                            ..
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
inzynieria materialowa                       1
journal of the indian botanical society      1
Name: Source, Length: 1015, dtype: int64

In [151]:
jour_df.apply(find_word, word = 'applied physics b', axis = 1);

0 applied physics b lasers and optics


In [152]:
genjoin({'applied physics b': 'applied physics b lasers and optics'})

There are 1810 Remaining Nulls
92.534% of the rows merged!


npj                                          9
journal of molecular catalysis a chemical    9
journal of physics energy                    9
optics for solar energy                      9
photochemical & photobiological sciences     8
                                            ..
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
inzynieria materialowa                       1
journal of the indian botanical society      1
Name: Source, Length: 1014, dtype: int64

In [153]:
jour_df.apply(find_word, word = 'physica status solid', axis = 1);

0 physica status solidi rapid research letters
0 physica status solidi (a) applications and materials science
0 physica status solidi (b) basic research


In [154]:
genjoin({'physica status solidiaapplied research': 'physica status solidi (a) applications and materials science'})

There are 1802 Remaining Nulls
92.567% of the rows merged!


npj                                          9
optics for solar energy                      9
journal of physics energy                    9
journal of molecular catalysis a chemical    9
photochemical & photobiological sciences     8
                                            ..
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
inzynieria materialowa                       1
journal of the indian botanical society      1
Name: Source, Length: 1013, dtype: int64

In [155]:
jour_df.apply(find_word, word = 'journal of optics', axis = 1);

0 journal of optics (united kingdom)
14 international journal of optics
0 journal of optics (india)


In [156]:
genjoin({'journal of optics': 'journal of optics (india)'})

There are 1796 Remaining Nulls
92.591% of the rows merged!


optics for solar energy                      9
npj                                          9
journal of molecular catalysis a chemical    9
journal of physics energy                    9
physica status solidi (c)                    8
                                            ..
inżynieria materiałowa                       1
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
journal of the indian botanical society      1
Name: Source, Length: 1012, dtype: int64

In [157]:
jour_df.apply(find_word, word = 'acta crystallographica section e', axis = 1);

0 acta crystallographica section e crystallographic communications


In [158]:
genjoin({ 'acta crystallographica section e structure reports online': 'acta crystallographica section e crystallographic communications'})

There are 1789 Remaining Nulls
92.620% of the rows merged!


journal of molecular catalysis a chemical    9
optics for solar energy                      9
npj                                          9
journal of physics energy                    9
photochemical & photobiological sciences     8
                                            ..
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
inzynieria materialowa                       1
journal of the indian botanical society      1
Name: Source, Length: 1011, dtype: int64

In [159]:
genjoin()[genjoin() > 5]

There are 1789 Remaining Nulls
92.620% of the rows merged!
There are 1789 Remaining Nulls
92.620% of the rows merged!


journal of molecular catalysis a chemical                                9
optics for solar energy                                                  9
npj                                                                      9
journal of physics energy                                                9
photochemical & photobiological sciences                                 8
physics procedia                                                         8
physica status solidi (c)                                                8
physica status solidi c                                                  7
european physical journalapplied physics                                 7
ecomat                                                                   7
acs biomaterials science & engineering                                   7
reaction kinetics and catalysis letters                                  7
optics & laser technology                                                7
josa b                   

In [160]:
jour_df.apply(find_word, word = 'physica status solidi', axis = 1);

0 physica status solidi rapid research letters
0 physica status solidi (a) applications and materials science
0 physica status solidi (b) basic research


Need to manually add `Physica Status Solidi (C) Current Topics in Solid State Physics`

In [161]:
id_ = jour_df['id'].values[-1] + 1
title = 'Physica Status Solidi (C) Current Topics in Solid State Physics'.lower()
type_ = 'journal'
SJR = 0.21
H_index = int(49)
country = 'Germany'
region = 'Western Europe'

jour_df.loc[jour_df.index.max() + 1, :] = [id_, title, type_, SJR, H_index, country, region]

In [162]:
jour_df.apply(find_word, word = 'physica status solidi', axis = 1);

0 physica status solidi rapid research letters
0 physica status solidi (a) applications and materials science
0 physica status solidi (b) basic research
0 physica status solidi (c) current topics in solid state physics


In [163]:
genjoin({ 'physica status solidi (c)': 'physica status solidi (c) current topics in solid state physics'})

There are 1781 Remaining Nulls
92.653% of the rows merged!


journal of molecular catalysis a chemical    9
journal of physics energy                    9
optics for solar energy                      9
npj                                          9
photochemical & photobiological sciences     8
                                            ..
inżynieria materiałowa                       1
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
journal of the indian botanical society      1
Name: Source, Length: 1010, dtype: int64

In [164]:
genjoin({ 'physica status solidi c': 'physica status solidi (c) current topics in solid state physics'})

There are 1774 Remaining Nulls
92.682% of the rows merged!


optics for solar energy                      9
journal of physics energy                    9
journal of molecular catalysis a chemical    9
npj                                          9
photochemical & photobiological sciences     8
                                            ..
inżynieria materiałowa                       1
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
journal of the indian botanical society      1
Name: Source, Length: 1009, dtype: int64

In [165]:
jour_df.apply(find_word, word = 'journal of molecular catalysis', axis = 1);

0 journal of molecular catalysis


In [166]:
genjoin({ 'journal of molecular catalysis a chemical ': 'journal of molecular catalysis'})

There are 1774 Remaining Nulls
92.682% of the rows merged!


optics for solar energy                      9
journal of physics energy                    9
journal of molecular catalysis a chemical    9
npj                                          9
photochemical & photobiological sciences     8
                                            ..
inżynieria materiałowa                       1
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
journal of the indian botanical society      1
Name: Source, Length: 1009, dtype: int64

In [167]:
jour_df.apply(find_word, word = 'reaction kinetics', axis = 1);

0 reaction kinetics, mechanisms and catalysis
12 progress in reaction kinetics and mechanism


In [168]:
genjoin({ 'reaction kinetics and catalysis letters': 'reaction kinetics, mechanisms and catalysis'})

There are 1767 Remaining Nulls
92.711% of the rows merged!


optics for solar energy                      9
journal of molecular catalysis a chemical    9
journal of physics energy                    9
npj                                          9
physics procedia                             8
                                            ..
inżynieria materiałowa                       1
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
journal of the indian botanical society      1
Name: Source, Length: 1008, dtype: int64

In [169]:
genjoin()[genjoin() >= 5]

There are 1767 Remaining Nulls
92.711% of the rows merged!
There are 1767 Remaining Nulls
92.711% of the rows merged!


optics for solar energy                                                  9
journal of molecular catalysis a chemical                                9
journal of physics energy                                                9
npj                                                                      9
physics procedia                                                         8
photochemical & photobiological sciences                                 8
ecomat                                                                   7
acs biomaterials science & engineering                                   7
optics & laser technology                                                7
european physical journalapplied physics                                 7
josa b                                                                   7
advances in resist materials and processing technology xxx               6
journal of nepal physical society                                        6
energy & environmental ma

In [170]:
jour_df.apply(find_word, word = 'interfaces', axis = 1);

24 acs applied materials & interfaces
13 surfaces and interfaces
10 composite interfaces
34 tribology materials, surfaces and interfaces
19 advanced materials interfaces
14 braincomputer interfaces
27 colloids and surfaces b biointerfaces
13 colloids and interfaces
0 interfaces and free boundaries


In [171]:
genjoin({ 'acs appl. mater. interfaces': 'acs applied materials & interfaces'})

There are 1762 Remaining Nulls
92.732% of the rows merged!


optics for solar energy                      9
journal of molecular catalysis a chemical    9
journal of physics energy                    9
npj                                          9
physics procedia                             8
                                            ..
inżynieria materiałowa                       1
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
journal of the indian botanical society      1
Name: Source, Length: 1007, dtype: int64

In [172]:
jour_df.apply(find_word, word = 'biomaterials', axis = 1);

0 biomaterials
0 biomaterials research
0 biomaterials science
4 acs biomaterials science and engineering
13 regenerative biomaterials
22 journal of functional biomaterials
25 international journal of biomaterials
56 journal of biomedical materials research part b applied biomaterials
11 journal of biomaterials science, polymer edition
11 journal of biomaterials applications
59 international journal of polymeric materials and polymeric biomaterials
19 journal of applied biomaterials and functional materials
32 bioinspired, biomimetic and nanobiomaterials
34 international journal of nano and biomaterials
10 paper and biomaterials
24 journal of biomimetics, biomaterials and biomedical engineering


In [173]:
genjoin({ 'acs biomaterials science & engineering': 'acs biomaterials science and engineering'})

There are 1755 Remaining Nulls
92.761% of the rows merged!


journal of molecular catalysis a chemical    9
journal of physics energy                    9
optics for solar energy                      9
npj                                          9
physics procedia                             8
                                            ..
inżynieria materiałowa                       1
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
journal of the indian botanical society      1
Name: Source, Length: 1006, dtype: int64

In [174]:
genjoin()[genjoin() >= 5]

There are 1755 Remaining Nulls
92.761% of the rows merged!
There are 1755 Remaining Nulls
92.761% of the rows merged!


journal of molecular catalysis a chemical                                9
journal of physics energy                                                9
optics for solar energy                                                  9
npj                                                                      9
physics procedia                                                         8
photochemical & photobiological sciences                                 8
ecomat                                                                   7
josa b                                                                   7
optics & laser technology                                                7
european physical journalapplied physics                                 7
small structures                                                         6
materials & design                                                       6
engineering reports                                                      6
clinical pharmacology and

In [175]:
jour_df.apply(find_word, word = 'physical review b', axis = 1);

0 physical review b


In [176]:
genjoin({ 'physical review b condensed matter and materials physics': 'physical review b'})

There are 1750 Remaining Nulls
92.781% of the rows merged!


journal of physics energy                    9
optics for solar energy                      9
journal of molecular catalysis a chemical    9
npj                                          9
physics procedia                             8
                                            ..
inżynieria materiałowa                       1
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
journal of the indian botanical society      1
Name: Source, Length: 1005, dtype: int64

In [177]:
jour_df.apply(find_word, word = 'materials and design', axis = 1);

0 materials and design


In [178]:
genjoin({ 'materials & design': 'materials and design'})

There are 1744 Remaining Nulls
92.806% of the rows merged!


npj                                          9
journal of molecular catalysis a chemical    9
journal of physics energy                    9
optics for solar energy                      9
photochemical & photobiological sciences     8
                                            ..
inżynieria materiałowa                       1
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
journal of the indian botanical society      1
Name: Source, Length: 1004, dtype: int64

In [179]:
jour_df.apply(find_word, word = 'molecular systems design', axis = 1);

0 molecular systems design and engineering


In [180]:
genjoin({ 'molecular systems design & engineering': 'molecular systems design and engineering'})

There are 1739 Remaining Nulls
92.827% of the rows merged!


npj                                          9
journal of molecular catalysis a chemical    9
journal of physics energy                    9
optics for solar energy                      9
photochemical & photobiological sciences     8
                                            ..
inżynieria materiałowa                       1
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
journal of the indian botanical society      1
Name: Source, Length: 1003, dtype: int64

In [181]:
genjoin()[genjoin() > 3]

There are 1739 Remaining Nulls
92.827% of the rows merged!
There are 1739 Remaining Nulls
92.827% of the rows merged!


npj                                               9
journal of molecular catalysis a chemical         9
journal of physics energy                         9
optics for solar energy                           9
photochemical & photobiological sciences          8
                                                 ..
faseb journal                                     4
energy storage                                    4
journal of physical chemistry bcondensed phase    4
indian journal of agronomy                        4
phys. chem. chem. phys                            4
Name: Source, Length: 93, dtype: int64

In [182]:
jour_df.apply(find_word, word = 'chemical physics', axis = 1);

11 journal of chemical physics
19 physical chemistry chemical physics
0 chemical physics letters
0 chemical physics
0 chemical physics letters x
19 chinese journal of chemical physics


In [183]:
genjoin({ 'phys. chem. chem. phys': 'physical chemistry chemical physics'})

There are 1735 Remaining Nulls
92.843% of the rows merged!


npj                                          9
journal of molecular catalysis a chemical    9
journal of physics energy                    9
optics for solar energy                      9
photochemical & photobiological sciences     8
                                            ..
inżynieria materiałowa                       1
solid state ionics ii volume                 1
évolution                                    1
fuel cells (online)                          1
journal of the indian botanical society      1
Name: Source, Length: 1002, dtype: int64

In [184]:
genjoin({ 'journal of molecular catalysis a chemical': 'journal of molecular catalysis'})

There are 1726 Remaining Nulls
92.880% of the rows merged!


journal of physics energy                   9
optics for solar energy                     9
npj                                         9
photochemical & photobiological sciences    8
physics procedia                            8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 1001, dtype: int64

A lot of the unassigned rows look to be abbreviations. Let's try to pull those out

In [185]:
import re

for i in genjoin().index:
    words = re.findall(r'\w+\.\s+\w+\.', i)
    if len(words) != 0:
        print(i)

There are 1726 Remaining Nulls
92.880% of the rows merged!
int. j. bioinorg. hybr. nanomater
phys. rev. lett.
adv. funct. mater
semicond. phys. quantum electron. optoelectron
appl. phys. lett
indian j. sci. technol
acs appl. mater. & interface
ukr. j. phys. opt
24th eur. photov. solar energy conf
int. j. electrochem. sci
appl. catal. b env
phys. chem. chem. phys.
bull. chem. soc. jpn
acc. chem. res
mod. phys. letts. b
chem. eng. trans
int. j. chem. res.
appl. phys. lett.
int. j. eng. res. appl
int. j. adv. res. sci. eng
am. j. polym. sci
phys. rev. materials
ieice technical report; ieice tech. rep.
int. j. sci. res.
int. j. innov. sci. eng. technol
sci. adv. today
int. j. innov. res. sci. eng. technol.
int. j. mech. mechatronics eng. ijmmeijens
bull. mineral. petrol. geochem
nat. chem.
int. j. chem. sci
indian j. sci. res.
intl. j. current microbiology appl. sci
int. res. j. pharm
journal of wuhan university of technologymater. sci. ed.
appl. surf. sci.
phys. rev. b condens. matter;(un

In [186]:
jour_df.apply(find_word, word = 'physica status solidi (a)', axis = 1);

0 physica status solidi (a) applications and materials science


In [187]:
genjoin({ 'phys. stat. sol.(b)': 'physica status solidi (b) basic research'})

There are 1725 Remaining Nulls
92.885% of the rows merged!


journal of physics energy                   9
optics for solar energy                     9
npj                                         9
photochemical & photobiological sciences    8
physics procedia                            8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 1000, dtype: int64

In [188]:
genjoin({ 'phys. stat. sol. (a)': 'physica status solidi (a) applications and materials science'})

There are 1724 Remaining Nulls
92.889% of the rows merged!


npj                                         9
optics for solar energy                     9
journal of physics energy                   9
physics procedia                            8
photochemical & photobiological sciences    8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 999, dtype: int64

In [189]:
jour_df.apply(find_word, word = 'angew', axis = 1);

17 agit journal fur angewandte geoinformatik
21 zamm zeitschrift fur angewandte mathematik und mechanik
0 angewandte chemie international edition
16 zeitschrift fur angewandte mathematik und physik


In [190]:
genjoin({ 'angew. chem. int. ed': 'angewandte chemie international edition'})

There are 1723 Remaining Nulls
92.893% of the rows merged!


journal of physics energy                   9
optics for solar energy                     9
npj                                         9
photochemical & photobiological sciences    8
physics procedia                            8
                                           ..
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
inzynieria materialowa                      1
journal of the indian botanical society     1
Name: Source, Length: 998, dtype: int64

In [191]:
jour_df.apply(find_word, word = 'physical review', axis = 1);

0 physical review b
0 physical review materials
0 physical review fluids
0 physical review x
0 physical review letters
0 physical review d
0 physical review applied
0 physical review c
0 physical review a
0 physical review physics education research
0 physical review accelerators and beams
0 physical review e


In [192]:
genjoin({'phys. rev. b condens. matter;(united states)':'physical review b'})

There are 1722 Remaining Nulls
92.897% of the rows merged!


journal of physics energy                   9
npj                                         9
optics for solar energy                     9
physics procedia                            8
photochemical & photobiological sciences    8
                                           ..
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
inzynieria materialowa                      1
journal of the indian botanical society     1
Name: Source, Length: 997, dtype: int64

In [193]:
jour_df.apply(find_word, word = 'surface', axis = 1);

0 surface science reports
12 progress in surface science
8 applied surface science
0 surface and coatings technology
0 surfaces and interfaces
51 journal of vacuum science and technology a vacuum, surfaces and films
0 surface engineering
21 tribology materials, surfaces and interfaces
0 surface science
8 applied surface science advances
0 surface science spectra
0 surface innovations
0 surface and interface analysis
0 surface topography metrology and properties
11 studies in surface science and catalysis
47 protection of metals and physical chemistry of surfaces
14 soldering and surface mount technology
0 surface review and letters
0 surface engineering and applied electrochemistry
25 international journal of surface science and engineering
12 ejournal of surface science and nanotechnology
3 biosurface and biotribology
11 journal of surface investigation
61 international journal of computational materials science and surface engineering
25 international journal of surface engineering a

In [194]:
genjoin({'appl. surf. sci':'applied surface science'})

There are 1721 Remaining Nulls
92.901% of the rows merged!


optics for solar energy                     9
journal of physics energy                   9
npj                                         9
photochemical & photobiological sciences    8
physics procedia                            8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 996, dtype: int64

In [195]:
jour_df.apply(find_word, word = 'physical review', axis = 1);

0 physical review b
0 physical review materials
0 physical review fluids
0 physical review x
0 physical review letters
0 physical review d
0 physical review applied
0 physical review c
0 physical review a
0 physical review physics education research
0 physical review accelerators and beams
0 physical review e


In [196]:
genjoin({'phys. rev. lett.':'physical review letters'})

There are 1717 Remaining Nulls
92.918% of the rows merged!


npj                                         9
optics for solar energy                     9
journal of physics energy                   9
physics procedia                            8
photochemical & photobiological sciences    8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 995, dtype: int64

In [197]:
jour_df.apply(find_word, word = 'functional materials', axis = 1);

9 advanced functional materials
5 multifunctional materials
36 journal of applied biomaterials and functional materials
0 functional materials letters
32 eurasian journal of physics and functional materials
0 functional materials
28 gongneng cailiao/journal of functional materials


In [198]:
genjoin({'adv. funct. mater':'advanced functional materials'})

There are 1713 Remaining Nulls
92.934% of the rows merged!


optics for solar energy                     9
npj                                         9
journal of physics energy                   9
photochemical & photobiological sciences    8
physics procedia                            8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 994, dtype: int64

In [199]:
jour_df.apply(find_word, word = 'semicond', axis = 1);

21 ieee transactions on semiconductor manufacturing
21 materials science in semiconductor processing
11 journal of semiconductors
0 semiconductor science and technology
32 ieee international symposium on semiconductor manufacturing conference, proceedings
0 semiconductors
11 journal of semiconductor technology and science
0 semiconductor physics, quantum electronics and optoelectronics
37 conference digest ieee international semiconductor laser conference
19 bandaoti guangdian/semiconductor optoelectronics
8 silicon semiconductor


In [200]:
genjoin({'semicond. phys. quantum electron. optoelectron':'semiconductor physics, quantum electronics and optoelectronics'})

There are 1710 Remaining Nulls
92.947% of the rows merged!


optics for solar energy                     9
journal of physics energy                   9
npj                                         9
photochemical & photobiological sciences    8
physics procedia                            8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 993, dtype: int64

In [201]:
jour_df.apply(find_word, word = 'applied phys', axis = 1);

18 journal physics d applied physics
8 current applied physics
0 applied physics a materials science and processing
4 epj applied physics
0 applied physics express
20 japanese journal of applied physics
36 johns hopkins apl technical digest (applied physics laboratory)
0 applied physics reviews
0 applied physics letters
11 journal of applied physics
0 applied physics b lasers and optics
27 journal of theoretical and applied physics
27 indian journal of pure and applied physics
0 applied physics


In [202]:
genjoin({'appl. phys. lett':'applied physics letters'})

There are 1707 Remaining Nulls
92.959% of the rows merged!


optics for solar energy                     9
journal of physics energy                   9
npj                                         9
photochemical & photobiological sciences    8
physics procedia                            8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 992, dtype: int64

In [203]:
jour_df.apply(find_word, word = 'ukr', axis = 1);

0 ukrainian journal of physics
0 ukrainian journal of physical optics


In [204]:
genjoin({'ukr. j. phys. opt':'ukrainian journal of physical optics'})

There are 1705 Remaining Nulls
92.967% of the rows merged!


npj                                         9
optics for solar energy                     9
journal of physics energy                   9
physics procedia                            8
photochemical & photobiological sciences    8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 991, dtype: int64

In [205]:
import re

for i in genjoin().index:
    words = re.findall(r'\w+\.\s+\w+\.', i)
    if len(words) != 0:
        print(i)

There are 1705 Remaining Nulls
92.967% of the rows merged!
int. j. bioinorg. hybr. nanomater
acs appl. mater. & interface
24th eur. photov. solar energy conf
am. j. polym. sci
appl. catal. b env
int. j. electrochem. sci
phys. chem. chem. phys.
bull. chem. soc. jpn
acc. chem. res
mod. phys. letts. b
int. j. chem. res.
chem. eng. trans
indian j. sci. technol
appl. phys. lett.
int. j. eng. res. appl
int. j. adv. res. sci. eng
ieice technical report; ieice tech. rep.
phys. rev. materials
int. j. sci. res.
int. j. innov. sci. eng. technol
int. j. mech. mechatronics eng. ijmmeijens
sci. adv. today
nat. chem.
bull. mineral. petrol. geochem
int. j. innov. res. sci. eng. technol.
int. j. chem. sci
indian j. sci. res.
intl. j. current microbiology appl. sci
int. res. j. pharm
journal of wuhan university of technologymater. sci. ed.
appl. surf. sci.
acta phys. chim. sin
phys. rev. lett
bull. soc. chim. fr.;(france)
stud. surf. sci. catal
nouv. j. chim.;(france)
jpn. j. appl. phys.
int. j. of adva

In [206]:
jour_df.apply(find_word, word = '& interface', axis = 1);

22 acs applied materials & interfaces


In [207]:
genjoin({'acs appl. mater. & interface':'acs applied materials & interfaces'})

There are 1703 Remaining Nulls
92.975% of the rows merged!


npj                                         9
optics for solar energy                     9
journal of physics energy                   9
physics procedia                            8
photochemical & photobiological sciences    8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 990, dtype: int64

In [208]:
jour_df.apply(find_word, word = 'applied catalysis', axis = 1);

0 applied catalysis b environmental
0 applied catalysis a general


In [209]:
genjoin({'appl. catal. b env':'applied catalysis b environmental'})

There are 1701 Remaining Nulls
92.984% of the rows merged!


optics for solar energy                     9
npj                                         9
journal of physics energy                   9
physics procedia                            8
photochemical & photobiological sciences    8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 989, dtype: int64

In [210]:
jour_df.apply(find_word, word = 'electrochem', axis = 1);

0 electrochemical energy reviews
15 journal of the electrochemical society
19 journal of applied electrochemistry
23 journal of solid state electrochemistry
11 journal of electrochemical energy conversion and storage
11 journal of electrochemical science and engineering
32 surface engineering and applied electrochemistry
29 journal of new materials for electrochemical systems
18 current topics in electrochemistry
19 current opinion in electrochemistry
0 electrochemistry communications
4 chemelectrochem
3 bioelectrochemistry
4 spr electrochemistry
0 electrochemical society interface
11 journal of electrochemical science and technology
0 electrochemistry
25 international journal of electrochemical science
19 russian journal of electrochemistry
29 analytical and bioanalytical electrochemistry


In [211]:
genjoin({'int. j. electrochem. sci':'international journal of electrochemical science'})

There are 1699 Remaining Nulls
92.992% of the rows merged!


npj                                         9
journal of physics energy                   9
optics for solar energy                     9
physics procedia                            8
photochemical & photobiological sciences    8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 988, dtype: int64

In [212]:
genjoin({'phys. chem. chem. phys.':'physical chemistry chemical physics'})

There are 1697 Remaining Nulls
93.000% of the rows merged!


optics for solar energy                     9
journal of physics energy                   9
npj                                         9
photochemical & photobiological sciences    8
physics procedia                            8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 987, dtype: int64

In [213]:
jour_df.apply(find_word, word = 'accounts', axis = 1);

0 accounts of chemical research
22 theoretical chemistry accounts


In [214]:
genjoin({'acc. chem. res':'accounts of chemical research'})

There are 1695 Remaining Nulls
93.008% of the rows merged!


npj                                         9
journal of physics energy                   9
optics for solar energy                     9
photochemical & photobiological sciences    8
physics procedia                            8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 986, dtype: int64

In [215]:
jour_df.apply(find_word, word = 'modern phys', axis = 1);

11 reviews of modern physics
25 international journal of modern physics d
25 international journal of modern physics a
89 studies in history and philosophy of science part b studies in history and philosophy of modern physics
0 modern physics letters a
46 international journal of geometric methods in modern physics
0 modern physics letters b
25 international journal of modern physics e
25 international journal of modern physics c
25 international journal of modern physics b
19 springer tracts in modern physics


In [216]:
genjoin({'mod. phys. letts. b':'modern physics letters b'})

There are 1693 Remaining Nulls
93.017% of the rows merged!


optics for solar energy                     9
npj                                         9
journal of physics energy                   9
photochemical & photobiological sciences    8
physics procedia                            8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 985, dtype: int64

In [217]:
jour_df.apply(find_word, word = 'chemical research', axis = 1);

12 accounts of chemical research
0 chemical research in chinese universities
11 journal of chemical research
15 trends in phytochemical research


In [218]:
genjoin({'int. j. chem. res.':'journal of chemical research'})

There are 1691 Remaining Nulls
93.025% of the rows merged!


npj                                         9
journal of physics energy                   9
optics for solar energy                     9
physics procedia                            8
photochemical & photobiological sciences    8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 984, dtype: int64

In [219]:
jour_df.apply(find_word, word = 'engineering transactions', axis = 1);

0 engineering transactions
25 international journal of engineering transactions c aspects
9 chemical engineering transactions


In [220]:
genjoin({'chem. eng. trans':'chemical engineering transactions'})

There are 1689 Remaining Nulls
93.033% of the rows merged!


npj                                         9
journal of physics energy                   9
optics for solar energy                     9
physics procedia                            8
photochemical & photobiological sciences    8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 983, dtype: int64

---

In [221]:
import re

for i in genjoin().index:
    words = re.findall(r'\w+\.\s+\w+\.', i)
    if len(words) != 0:
        print(i)

There are 1689 Remaining Nulls
93.033% of the rows merged!
int. j. bioinorg. hybr. nanomater
24th eur. photov. solar energy conf
am. j. polym. sci
indian j. sci. technol
int. j. eng. res. appl
appl. phys. lett.
int. j. adv. res. sci. eng
bull. chem. soc. jpn
int. j. innov. sci. eng. technol
phys. rev. materials
int. j. sci. res.
ieice technical report; ieice tech. rep.
int. j. innov. res. sci. eng. technol.
sci. adv. today
int. j. mech. mechatronics eng. ijmmeijens
nat. chem.
int. j. chem. sci
bull. mineral. petrol. geochem
indian j. sci. res.
intl. j. current microbiology appl. sci
int. res. j. pharm
journal of wuhan university of technologymater. sci. ed.
appl. surf. sci.
acta phys. chim. sin
phys. rev. lett
bull. soc. chim. fr.;(france)
stud. surf. sci. catal
nouv. j. chim.;(france)
jpn. j. appl. phys.
int. j. of advanced applied physics research
chem. eur. j.
rev. comput. mol. sci
int. j. health sci. res


In [222]:
jour_df.apply(find_word, word = 'review letters', axis = 1);

9 physical review letters


In [223]:
genjoin({'nat. chem':'nature chem'})

There are 1689 Remaining Nulls
93.033% of the rows merged!


npj                                         9
journal of physics energy                   9
optics for solar energy                     9
physics procedia                            8
photochemical & photobiological sciences    8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 983, dtype: int64

In [224]:
genjoin({'phys. rev. lett':'physical review letters'})

There are 1688 Remaining Nulls
93.037% of the rows merged!


optics for solar energy                     9
journal of physics energy                   9
npj                                         9
physics procedia                            8
photochemical & photobiological sciences    8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 982, dtype: int64

In [225]:
jour_df.apply(find_word, word = 'surface science', axis = 1);

0 surface science reports
12 progress in surface science
8 applied surface science
0 surface science
8 applied surface science advances
0 surface science spectra
11 studies in surface science and catalysis
25 international journal of surface science and engineering
12 ejournal of surface science and nanotechnology
11 journal of surface science and technology


In [226]:
genjoin({'appl. surf. sci.':'applied surface science'})

There are 1687 Remaining Nulls
93.042% of the rows merged!


npj                                         9
journal of physics energy                   9
optics for solar energy                     9
physics procedia                            8
photochemical & photobiological sciences    8
                                           ..
inżynieria materiałowa                      1
solid state ionics ii volume                1
évolution                                   1
fuel cells (online)                         1
journal of the indian botanical society     1
Name: Source, Length: 981, dtype: int64

In [227]:
'phys. rev. materials'
jour_df.apply(find_word, word = 'review materials', axis = 1);

9 physical review materials


Really tired...just gonna call it. > 93% is good

In [229]:
join_df1 = pd.merge(left = paper_df, right = jour_df, left_on = 'Source', right_on = 'Title')
join_df1

,Cites,Title_x,Year,Source,GSRank,Type_x,ECC,CitesPerYear,CitesPerAuthor,AuthorCount,...,first_auth,second_auth,third_auth,id,Title_y,Type_y,SJR,H index,Country,Region
0,4186,low trap-state density and long carrier diffus...,2015.0,science,1,Journal article,4186,523.25,523,8,...,d shi,v adinolfi,r comin,6405.0,science,journal,14.589,1229.0,United States,Northern America
1,1951,efficient and stable solution-processed planar...,2017.0,science,4,Journal article,1951,325.17,279,7,...,h tan,a jain,o voznyy,6405.0,science,journal,14.589,1229.0,United States,Northern America
2,1756,homogeneously dispersed multimetal oxygen-evol...,2016.0,science,7,Journal article,1756,250.86,251,7,...,b zhang,x zheng,o voznyy,6405.0,science,journal,14.589,1229.0,United States,Northern America
3,1343,co2 electroreduction to ethylene via hydroxide...,2018.0,science,11,Journal article,1343,268.60,224,6,...,ct dinh,t burdyny,mg kibria,6405.0,science,journal,14.589,1229.0,United States,Northern America
4,1168,what would it take for renewably powered elect...,2019.0,science,14,Journal article,1168,292.00,195,6,...,p de luna,c hahn,d higgins,6405.0,science,journal,14.589,1229.0,United States,Northern America
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22525,2,molecular insights into water vapor adsorption...,2021.0,international journal of heat and mass transfer,6,Journal article,2,1.00,0,7,...,s lin,c chen,l zhao,3127.0,international journal of heat and mass transfer,journal,1.461,224.0,United Kingdom,Western Europe
22526,6,nanoscale siox tunnel oxide deposition techniq...,2021.0,transactions on electrical and electronic mate...,4,Journal article,6,3.00,1,5,...,sp padi,mq khokhar,s chowdhury,891.0,transactions on electrical and electronic mate...,journal,0.276,22.0,South Korea,Asiatic Region
22527,34,rapid biodegradation of renewable polyurethane...,2020.0,bioresource technology reports,2,Journal article,34,11.33,6,6,...,nr gunawan,m tessman,ac schreiman,1708.0,bioresource technology reports,journal,0.837,27.0,United Kingdom,Western Europe
22528,26,pitch-rotational manipulation of single cells ...,2020.0,biomedical optics express,1,Journal article,26,8.67,3,8,...,s kumar,m gunaseelan,r vaippully,5970.0,biomedical optics express,journal,1.120,93.0,United States,Northern America


In [230]:
join_df2 = join_df1.drop(['id', 'Title_y' , 'Type_x'], axis = 1)

Reset column headers

In [232]:
join_df2.columns

Index(['Cites', 'Title_x', 'Year', 'Source', 'GSRank', 'ECC', 'CitesPerYear',
       'CitesPerAuthor', 'AuthorCount', 'query_id', 'first_auth',
       'second_auth', 'third_auth', 'Type_y', 'SJR', 'H index', 'Country',
       'Region'],
      dtype='object')

In [240]:
join_df2.columns = ['Cites', 'Title', 'Year', 'Source', 'GSRank', 'ECC', 'CitesPerYear',
       'CitesPerAuthor', 'AuthorCount', 'query_id', 'first_auth',
       'second_auth', 'third_auth', 'ArticleType', 'SJR', 'Hindex', 'SourceCountry',
       'SourceRegion']

In [241]:
join_df2.head()

,Cites,Title,Year,Source,GSRank,ECC,CitesPerYear,CitesPerAuthor,AuthorCount,query_id,first_auth,second_auth,third_auth,ArticleType,SJR,Hindex,SourceCountry,SourceRegion
0,4186,low trap-state density and long carrier diffus...,2015.0,science,1,4186,523.25,523,8,0.0,d shi,v adinolfi,r comin,journal,14.589,1229.0,United States,Northern America
1,1951,efficient and stable solution-processed planar...,2017.0,science,4,1951,325.17,279,7,0.0,h tan,a jain,o voznyy,journal,14.589,1229.0,United States,Northern America
2,1756,homogeneously dispersed multimetal oxygen-evol...,2016.0,science,7,1756,250.86,251,7,0.0,b zhang,x zheng,o voznyy,journal,14.589,1229.0,United States,Northern America
3,1343,co2 electroreduction to ethylene via hydroxide...,2018.0,science,11,1343,268.60,224,6,0.0,ct dinh,t burdyny,mg kibria,journal,14.589,1229.0,United States,Northern America
4,1168,what would it take for renewably powered elect...,2019.0,science,14,1168,292.00,195,6,0.0,p de luna,c hahn,d higgins,journal,14.589,1229.0,United States,Northern America


Want to double check null values

In [242]:
join_df2.isna().sum()

Cites               0
Title               2
Year                0
Source              0
GSRank              0
ECC                 0
CitesPerYear        0
CitesPerAuthor      0
AuthorCount         0
query_id            0
first_auth          0
second_auth         0
third_auth        812
ArticleType         0
SJR                 0
Hindex              0
SourceCountry       0
SourceRegion        0
dtype: int64

In [243]:
join_df2[join_df2['Title'].isna()]

,Cites,Title,Year,Source,GSRank,ECC,CitesPerYear,CitesPerAuthor,AuthorCount,query_id,first_auth,second_auth,third_auth,ArticleType,SJR,Hindex,SourceCountry,SourceRegion
19504,0,NaN,2019.0,science china materials,301,0,0.0,0,11,12.0,z zhang,f sun,z zhu,journal,1.603,41.0,China,Asiatic Region
19506,0,NaN,2020.0,science china materials,249,0,0.0,0,4,17.0,l wang,g dai,l deng,journal,1.603,41.0,China,Asiatic Region


Weird nulls in `Title`, should investigate later. But no nulls in `SJR` or `H index`, so can officially change their data types. Let's also change that data type of `query_id`

In [248]:
# SJR to float
join_df2 = join_df2.assign(SJR = join_df2.apply(lambda x: float(x['SJR']), axis = 1))

#H index to int
join_df2 = join_df2.assign(Hindex = join_df2.apply(lambda x: int(x['Hindex']), axis = 1))

#query_id to int
join_df2 = join_df2.assign(query_id = join_df2.apply(lambda x: int(x['query_id']), axis = 1))

In [249]:
join_df2.head()

,Cites,Title,Year,Source,GSRank,ECC,CitesPerYear,CitesPerAuthor,AuthorCount,query_id,first_auth,second_auth,third_auth,ArticleType,SJR,Hindex,SourceCountry,SourceRegion
0,4186,low trap-state density and long carrier diffus...,2015.0,science,1,4186,523.25,523,8,0,d shi,v adinolfi,r comin,journal,14.589,1229,United States,Northern America
1,1951,efficient and stable solution-processed planar...,2017.0,science,4,1951,325.17,279,7,0,h tan,a jain,o voznyy,journal,14.589,1229,United States,Northern America
2,1756,homogeneously dispersed multimetal oxygen-evol...,2016.0,science,7,1756,250.86,251,7,0,b zhang,x zheng,o voznyy,journal,14.589,1229,United States,Northern America
3,1343,co2 electroreduction to ethylene via hydroxide...,2018.0,science,11,1343,268.60,224,6,0,ct dinh,t burdyny,mg kibria,journal,14.589,1229,United States,Northern America
4,1168,what would it take for renewably powered elect...,2019.0,science,14,1168,292.00,195,6,0,p de luna,c hahn,d higgins,journal,14.589,1229,United States,Northern America


Now let's save both the `join_df2` and `jour_df` data frame

In [251]:
import os
cleanPath = r'G:\My Drive\BrainStation\BrainStationCapstone\Scripts and Notebooks\Cleaned Data'
clean_file = os.path.join(cleanPath, '2023-03-04 Joining DFs_join_Df2.csv')
join_df2.to_csv(clean_file, index=False)


clean_file = os.path.join(cleanPath, '2023-03-04 Joining DFs_jour_df.csv')
jour_df.to_csv(clean_file, index=False)

----